## Cell 0 — Synthetic demo dataset (deep-research compliant)

This cell generates a **reproducible synthetic diary** to stress-test the full trigger-detection pipeline end-to-end.

It creates multiple users with realistic meal timing and coverage, injects a **ground-truth trigger family (DAIRY)** that appears more often in **hazard windows** *(T−L, T−e]* while respecting a **minimum lag**, and adds a **negative-control ingredient** expected to show no association.

**Outputs written (required downstream schema):**
- `../data/synthetic/user_meals.csv` — `user_id, meal_id, meal_time, ingredient`
- `../data/synthetic/user_symptoms.csv` — `user_id, symptom_time, symptom_severity`

It also prints **coverage + sanity diagnostics** (meal time range, example hazard windows, and hazard vs non-hazard dairy rates).


In [67]:
from pathlib import Path
from collections import defaultdict
from math import comb
import os
import re

import numpy as np
import pandas as pd


# -----------------------------
# Output paths
# -----------------------------
SYN_DIR = Path("../data/synthetic")
SYN_DIR.mkdir(parents=True, exist_ok=True)

MEALS_PATH = SYN_DIR / "user_meals.csv"
SYMPTOMS_PATH = SYN_DIR / "user_symptoms.csv"

# -----------------------------
# Reproducibility
# -----------------------------
rng = np.random.default_rng(7)

# -----------------------------
# Deep-research method knobs (match later pipeline defaults)
# -----------------------------
CASE_HOURS = 24            # L (hazard window length)
MIN_LAG_HOURS = 2          # e (minimum lag): hazard window is (T-L, T-e]
CONTROL_LOOKBACK_DAYS = 28 # controls searched within ~4 weeks
EPISODE_GAP_HOURS = 8      # cluster high-severity logs into episodes
EVENT_SEVERITY_THRESHOLD = 7

# Time matching (controls): same clock-time + same day-of-week is the main target
# This cell creates events with stable weekday and time-of-day patterns so this works well.

# -----------------------------
# Timeline: ensure > 28 days of meals BEFORE first event
# -----------------------------
start = pd.Timestamp("2025-11-15 00:00:00")
end   = pd.Timestamp("2026-01-20 23:59:00")
days = pd.date_range(start.normalize(), end.normalize(), freq="D")

# Stable daily schedule + small jitter (so time matching tolerance logic is testable later)
base_meal_times = [
    ("breakfast", pd.Timedelta(hours=8, minutes=10)),
    ("lunch",     pd.Timedelta(hours=13, minutes=5)),
    ("dinner",    pd.Timedelta(hours=19, minutes=30)),
]

def jitter_minutes(max_abs=10):
    return pd.Timedelta(minutes=int(rng.integers(-max_abs, max_abs + 1)))

# -----------------------------
# Ingredient pools (cleaned-style tokens)
# -----------------------------
BASE = [
    "rice", "chicken", "pure olive oil",
    "tomato", "onion", "pepper",
    "banana", "oats", "honey",
    "lettuce", "cucumber", "apple",
    "beans", "potato", "carrot",
]

# TRUE TRIGGER FAMILY (ground truth)
DAIRY = ["cheese", "cream", "butter", "milk", "yogurt"]

# Negative-control ingredient (should NOT be detected as trigger if method works)
NEG_CONTROL = ["cumin"]  # included at constant rate in and out of hazard windows

# Neutral extras (random noise)
EXTRAS = ["flour", "egg", "pasta", "bread"]

# -----------------------------
# Users: multiple users to prove "within-person" works
#   u1 = strong dairy trigger
#   u2 = weaker dairy trigger (harder detection)
#   u3 = no trigger (should not show dairy as significant)
# -----------------------------
USERS = [
    dict(user_id="u1", dairy_hazard_boost=0.75, dairy_baseline=0.15, n_events=5),
    dict(user_id="u2", dairy_hazard_boost=0.45, dairy_baseline=0.15, n_events=5),
    dict(user_id="u3", dairy_hazard_boost=0.15, dairy_baseline=0.15, n_events=5),  # null user
]

# -----------------------------
# Hazard window helpers: (T-L, T-e]
# -----------------------------
def hazard_window(t_event: pd.Timestamp):
    return (
        t_event - pd.Timedelta(hours=CASE_HOURS),
        t_event - pd.Timedelta(hours=MIN_LAG_HOURS),
    )

# -----------------------------
# Pick ingredients for a meal
# -----------------------------
def pick_meal_ingredients(meal_type: str, is_hazard_period: bool, dairy_baseline: float, dairy_hazard_boost: float):
    """
    Construct a realistic ingredient set.
    - Inject DAIRY more often in hazard windows to create ground-truth signal
    - Add a NEGATIVE CONTROL ingredient at constant rate
    """
    ings = set()

    # 2–3 baseline items
    ings |= set(rng.choice(BASE, size=int(rng.integers(2, 4)), replace=False))

    # negative control always at constant probability (no hazard effect)
    if rng.random() < 0.25:
        ings.add(rng.choice(NEG_CONTROL))

    # meal-type-specific tweaks
    if meal_type == "breakfast":
        ings.add(rng.choice(["banana", "oats", "apple", "honey"]))
        # dairy in breakfast: milk/yogurt
        p_dairy = dairy_baseline + (dairy_hazard_boost if is_hazard_period else 0.0)
        p_dairy = min(0.95, max(0.0, p_dairy))
        if rng.random() < p_dairy:
            ings.add(rng.choice(["milk", "yogurt"]))
        if rng.random() < 0.20:
            ings.add("egg")

    elif meal_type == "lunch":
        ings.add(rng.choice(["rice", "chicken", "tomato", "onion", "beans"]))
        if rng.random() < 0.30:
            ings.add("pure olive oil")
        # lunch dairy is rare (keeps effect concentrated in certain meals)
        p_dairy = 0.02 + (0.20 if is_hazard_period else 0.0)
        if rng.random() < p_dairy:
            ings.add(rng.choice(["cheese"]))

    elif meal_type == "dinner":
        if rng.random() < 0.25:
            ings.add(rng.choice(["pasta", "bread", "potato"]))
        p_dairy = dairy_baseline + (dairy_hazard_boost if is_hazard_period else 0.0)
        p_dairy = min(0.95, max(0.0, p_dairy))
        if rng.random() < p_dairy:
            ings.add(rng.choice(["cheese", "cream", "butter"]))

    # small chance of extra neutral item
    if rng.random() < 0.20:
        ings.add(rng.choice(EXTRAS))

    # cap size (avoid huge meals)
    if len(ings) > 7:
        ings = set(rng.choice(list(ings), size=7, replace=False))

    return sorted(ings)

# -----------------------------
# Event scheduling:
# Create events at similar clock-times and SAME DAY-OF-WEEK
# so your time-stratified matching (incl. weekday match) is meaningful.
# -----------------------------
def make_event_schedule(user_id: str, n_events: int):
    # pick a reference weekday by choosing a real date and reusing its weekday
    base_event = pd.Timestamp("2026-01-02 22:30:00")  # Friday
    # spread events ~every 7 days to preserve weekday structure for matching
    event_times = []
    for k in range(n_events):
        # +/- small minutes jitter to test tolerance logic later
        jitter = pd.Timedelta(minutes=int(rng.integers(-20, 21)))
        event_times.append(base_event + pd.Timedelta(days=7 * k) + jitter)

    # ensure all are within [start, end]
    event_times = [t for t in event_times if (t >= start + pd.Timedelta(days=30)) and (t <= end - pd.Timedelta(days=1))]
    return event_times

# -----------------------------
# Generate meals + symptoms
# -----------------------------
meal_rows = []
sym_rows = []

meal_counter = 1
P_SKIP_MEAL = 0.05

for u in USERS:
    user_id = u["user_id"]
    dairy_hazard_boost = float(u["dairy_hazard_boost"])
    dairy_baseline = float(u["dairy_baseline"])
    n_events = int(u["n_events"])

    # define severe episode onsets (ground truth)
    severe_episode_onsets = make_event_schedule(user_id, n_events)

    # precompute hazard windows for injection logic
    hazard_windows = [hazard_window(t0) for t0 in severe_episode_onsets]

    def is_in_any_hazard_window(ts: pd.Timestamp) -> bool:
        for a, b in hazard_windows:
            if (ts > a) and (ts <= b):
                return True
        return False

    # ---- Meals
    for d in days:
        for meal_type, offset in base_meal_times:
            if rng.random() < P_SKIP_MEAL:
                continue

            t = d + offset + jitter_minutes(10)

            if t < start or t > end:
                continue

            meal_id = f"{user_id}_m{meal_counter}"
            meal_counter += 1

            hazard_flag = is_in_any_hazard_window(t)
            ing_list = pick_meal_ingredients(
                meal_type=meal_type,
                is_hazard_period=hazard_flag,
                dairy_baseline=dairy_baseline,
                dairy_hazard_boost=dairy_hazard_boost,
            )

            for ing in ing_list:
                meal_rows.append({
                    "user_id": user_id,
                    "meal_id": meal_id,
                    "meal_time": t.strftime("%Y-%m-%d %H:%M:%S"),
                    "ingredient": ing,
                })

    # ---- Symptoms (episodes)
    # For episode clustering: 1–2 extra high-severity logs within EPISODE_GAP_HOURS
    for t0 in severe_episode_onsets:
        sym_rows.append({
            "user_id": user_id,
            "symptom_time": t0.strftime("%Y-%m-%d %H:%M:%S"),
            "symptom_severity": int(rng.integers(EVENT_SEVERITY_THRESHOLD, 10)),  # onset
        })
        if rng.random() < 0.70:
            sym_rows.append({
                "user_id": user_id,
                "symptom_time": (t0 + pd.Timedelta(hours=int(rng.integers(1, 6)))).strftime("%Y-%m-%d %H:%M:%S"),
                "symptom_severity": int(rng.integers(EVENT_SEVERITY_THRESHOLD, 10)),
            })
        if rng.random() < 0.35:
            sym_rows.append({
                "user_id": user_id,
                "symptom_time": (t0 + pd.Timedelta(hours=int(rng.integers(2, 8)))).strftime("%Y-%m-%d %H:%M:%S"),
                "symptom_severity": int(rng.integers(EVENT_SEVERITY_THRESHOLD, 10)),
            })

    # ---- Mild/moderate noise symptoms (should NOT create extra episodes)
    n_noise = 20
    total_minutes = int((end - start).total_seconds() / 60)
    noise_times = set()

    while len(noise_times) < n_noise:
        dt = start + pd.Timedelta(minutes=int(rng.integers(0, total_minutes)))
        # avoid noise too close to severe onsets (keep episodes clean)
        if all(abs((dt - t0).total_seconds()) > 6 * 3600 for t0 in severe_episode_onsets):
            noise_times.add(dt)

    for t in sorted(noise_times):
        sym_rows.append({
            "user_id": user_id,
            "symptom_time": t.strftime("%Y-%m-%d %H:%M:%S"),
            "symptom_severity": int(rng.integers(1, EVENT_SEVERITY_THRESHOLD)),  # below threshold
        })

meals_df = pd.DataFrame(meal_rows)
symptoms_df = pd.DataFrame(sym_rows).sort_values(["user_id", "symptom_time"]).reset_index(drop=True)

# -----------------------------
# Save
# -----------------------------
meals_df.to_csv(MEALS_PATH, index=False)
symptoms_df.to_csv(SYMPTOMS_PATH, index=False)

print("Wrote DEEP-RESEARCH-COMPLIANT demo files:")
print(" -", MEALS_PATH, "| rows:", len(meals_df),
      "| users:", meals_df["user_id"].nunique(),
      "| unique meals:", meals_df["meal_id"].nunique(),
      "| unique ingredients:", meals_df["ingredient"].nunique())
print(" -", SYMPTOMS_PATH, "| rows:", len(symptoms_df),
      "| users:", symptoms_df["user_id"].nunique(),
      "| high-severity logs (>=thr):", int((symptoms_df["symptom_severity"] >= EVENT_SEVERITY_THRESHOLD).sum()))

# -----------------------------
# Coverage diagnostics (critical for control eligibility)
# -----------------------------
meal_min = pd.to_datetime(meals_df["meal_time"]).min()
meal_max = pd.to_datetime(meals_df["meal_time"]).max()

print("\n Coverage check (for matched controls within ~4 weeks)")
print("Meal time range:", meal_min, "->", meal_max)
print("CONTROL_LOOKBACK_DAYS:", CONTROL_LOOKBACK_DAYS)

# Show hazard window definition
print("\n Hazard window spec used for injection:")
print("CASE_HOURS (L) =", CASE_HOURS, "| MIN_LAG_HOURS (e) =", MIN_LAG_HOURS)

# Show a few events + hazard windows per user (auditability)
print("\n Example event/hazard windows per user (first 2 each):")
for user_id in sorted(symptoms_df["user_id"].unique()):
    severe = symptoms_df[(symptoms_df["user_id"] == user_id) &
                         (symptoms_df["symptom_severity"] >= EVENT_SEVERITY_THRESHOLD)].copy()
    severe["symptom_time"] = pd.to_datetime(severe["symptom_time"])
    severe = severe.sort_values("symptom_time")

    # approximate episode onset candidates (we'll cluster later properly)
    onsets = list(severe["symptom_time"].head(2).values)
    print(f"\nUser {user_id}:")
    for t0 in onsets:
        t0 = pd.Timestamp(t0)
        hs, he = hazard_window(t0)
        print("  Event:", t0, "| hazard:", hs, "->", he)

# Ground-truth sanity: show dairy rates in hazard vs non-hazard for u1
# (rough check; later cells do the real inference)
u1_meals = meals_df[meals_df["user_id"] == "u1"].copy()
u1_meals["meal_time"] = pd.to_datetime(u1_meals["meal_time"])
u1_meals = u1_meals.drop_duplicates(["meal_id", "ingredient"])  # binary per meal per ingredient

# build u1 event times (from symptoms severe logs; not clustered here)
u1_sev = symptoms_df[(symptoms_df["user_id"] == "u1") &
                     (symptoms_df["symptom_severity"] >= EVENT_SEVERITY_THRESHOLD)].copy()
u1_sev["symptom_time"] = pd.to_datetime(u1_sev["symptom_time"])
u1_sev = u1_sev.sort_values("symptom_time")

# take first log of each week as crude event anchor for this diagnostic
u1_event_times = []
for t in u1_sev["symptom_time"]:
    if not u1_event_times or (t - u1_event_times[-1]) > pd.Timedelta(hours=EPISODE_GAP_HOURS):
        u1_event_times.append(t)

haz_windows = [hazard_window(t0) for t0 in u1_event_times]

def in_hazard(ts):
    for a, b in haz_windows:
        if (ts > a) and (ts <= b):
            return True
    return False

u1_meals["is_hazard"] = u1_meals["meal_time"].apply(in_hazard)
u1_meals["is_dairy"] = u1_meals["ingredient"].isin(DAIRY)

dairy_rate_h = u1_meals[u1_meals["is_hazard"]]["is_dairy"].mean()
dairy_rate_n = u1_meals[~u1_meals["is_hazard"]]["is_dairy"].mean()

print("\n Ground-truth injection sanity (u1):")
print("Approx dairy proportion in hazard meals:", round(float(dairy_rate_h), 3))
print("Approx dairy proportion outside hazard:", round(float(dairy_rate_n), 3))
print("Negative control ingredient:", NEG_CONTROL[0], "(should not become a trigger)")

display(meals_df.head(12))
display(symptoms_df.head(12))


Wrote DEEP-RESEARCH-COMPLIANT demo files:
 - ../data/synthetic/user_meals.csv | rows: 2233 | users: 3 | unique meals: 582 | unique ingredients: 25
 - ../data/synthetic/user_symptoms.csv | rows: 73 | users: 3 | high-severity logs (>=thr): 13

 Coverage check (for matched controls within ~4 weeks)
Meal time range: 2025-11-15 08:03:00 -> 2026-01-20 19:34:00
CONTROL_LOOKBACK_DAYS: 28

 Hazard window spec used for injection:
CASE_HOURS (L) = 24 | MIN_LAG_HOURS (e) = 2

 Example event/hazard windows per user (first 2 each):

User u1:
  Event: 2026-01-02 22:48:00 | hazard: 2026-01-01 22:48:00 -> 2026-01-02 20:48:00
  Event: 2026-01-02 23:48:00 | hazard: 2026-01-01 23:48:00 -> 2026-01-02 21:48:00

User u2:
  Event: 2026-01-02 22:44:00 | hazard: 2026-01-01 22:44:00 -> 2026-01-02 20:44:00
  Event: 2026-01-09 22:12:00 | hazard: 2026-01-08 22:12:00 -> 2026-01-09 20:12:00

User u3:
  Event: 2026-01-02 22:36:00 | hazard: 2026-01-01 22:36:00 -> 2026-01-02 20:36:00
  Event: 2026-01-03 01:36:00 | hazar

,user_id,meal_id,meal_time,ingredient
0,u1,u1_m1,2025-11-15 08:16:00,carrot
1,u1,u1_m1,2025-11-15 08:16:00,cumin
2,u1,u1_m1,2025-11-15 08:16:00,oats
3,u1,u1_m1,2025-11-15 08:16:00,onion
4,u1,u1_m2,2025-11-15 13:12:00,banana
5,u1,u1_m2,2025-11-15 13:12:00,potato
6,u1,u1_m2,2025-11-15 13:12:00,tomato
7,u1,u1_m3,2025-11-15 19:40:00,beans
8,u1,u1_m3,2025-11-15 19:40:00,cumin
9,u1,u1_m3,2025-11-15 19:40:00,lettuce


,user_id,symptom_time,symptom_severity
0,u1,2025-11-18 13:52:00,4
1,u1,2025-11-25 05:59:00,2
2,u1,2025-11-26 04:52:00,5
3,u1,2025-11-27 11:46:00,3
4,u1,2025-12-02 13:41:00,5
5,u1,2025-12-04 16:17:00,5
6,u1,2025-12-04 16:32:00,5
7,u1,2025-12-08 23:58:00,1
8,u1,2025-12-10 00:48:00,1
9,u1,2025-12-13 00:07:00,5


## Cell 1 — Trigger detection configuration (study design + parameters)

This cell defines the **full experimental design** and all hyperparameters for SafePlate’s trigger-detection pipeline, in a way that is **academically defensible** and still **MVP-feasible**.

**Core idea (self-matched case-crossover):**
- We treat each symptom episode onset as an **event time** and compare ingredient exposure in a **hazard window** *(T−L, T−e]* against **matched control windows** from earlier weeks for the *same person*.

**What gets configured here:**
- **Reproducibility:** fixed global seed for deterministic sampling.
- **Input/Output paths:** meals, symptoms, ingredient→family mapping, and output directory for report-ready CSVs.
- **Episode clustering:** merge multiple high-severity logs into a single flare episode to avoid double-counting.
- **Case window definition:** hazard length *(L)* and minimum lag *(e)* to reduce reverse-causality / implausible timing.
- **Control strategy:** time-stratified matched controls (time-of-day + day-of-week) with symptom-free eligibility rules.
- **Exposure definition:** binary exposure (ingredient appears at least once in the window) with an explicit boundary convention.
- **Inference settings:** pseudocount stabilization, minimum evidence threshold, and BH-FDR cutoff for flagged triggers.
- **Optional Bayesian layer:** a Beta posterior that provides an interpretable “belief” measure, intended as *additional uncertainty context*, not a replacement for FDR control.
- **Optional family pooling toggle:** conservative shrinkage option (off by default).

At the end, the cell prints a **parameter summary** so the notebook is auditable and easy to reference in the progress report.


In [68]:
# -----------------------------
# Reproducibility / determinism
# -----------------------------
GLOBAL_SEED = 42
rng = np.random.default_rng(GLOBAL_SEED)

# -----------------------------
# Input paths (relative to notebooks/)
# -----------------------------
# Use synthetic by default; swap to processed logs later (same schema)
DATA_DIR = Path("../data/synthetic")
MEALS_PATH = DATA_DIR / "user_meals.csv"
SYMPTOMS_PATH = DATA_DIR / "user_symptoms.csv"

# Ingredient family mapping (post-processing aggregation; do NOT change ingredients)
GROUPS_PATH = Path("../data/processed") / "ingredient_groups.csv"

# Output directory (report-ready CSVs)
OUT_DIR = Path("../data/processed") / "trigger_detection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Core study design parameters (Recommended Design)
# -----------------------------
# Episode clustering: merge high-severity symptom logs into episodes to avoid double-counting
EPISODE_GAP_HOURS = 8

# "Event" definition (episode onset) threshold.
# Keep fixed threshold for MVP; percentile fallback can be enabled later.
EVENT_SEVERITY_THRESHOLD = 7
USE_USER_PERCENTILE_FALLBACK = False
USER_PERCENTILE = 0.75

# Hazard / case window definition: (T−L, T−e]
CASE_HOURS = 24           # L (spec supports up to 24h)
MIN_LAG_HOURS = 2         # e (recommended to avoid implausible immediate causality / reverse timing)

# -----------------------------
# Control selection (time-stratified matched controls)
# -----------------------------
# We sample controls from the prior ~4 weeks, matched on time-of-day and day-of-week
CONTROL_LOOKBACK_DAYS = 28
N_CONTROLS_PER_EVENT = 4

# Matching criteria (deep-research compliant)
MATCH_DAY_OF_WEEK = True

# Time-of-day matching approach:
# - Primary: exact minute-of-day match (tolerance 0)
# - Fallback: widen tolerance (e.g., ±60 min) if too few candidates
TIME_MATCH_TOLERANCE_MIN = 0
TIME_MATCH_FALLBACK_MIN = 60

# Symptom-free eligibility for controls:
# Recommended MVP:
#   "no_episode_onset" => exclude windows containing an EPISODE ONSET (episode start_time)
# Stronger option (more conservative, optional):
#   "no_any_symptom"   => exclude windows containing ANY symptom log (even mild)
CONTROL_SYMPTOM_FREE_MODE = "no_episode_onset"
REQUIRE_SYMPTOM_FREE_CONTROLS = True

# If an event has < MIN_CONTROLS_REQUIRED eligible controls, drop it (avoid biased counts)
MIN_CONTROLS_REQUIRED = 1

# -----------------------------
# Exposure definition (binary)
# -----------------------------
# Ingredient is "exposed" in a window if it appears in ANY meal in that window.
# Boundary convention (audit-critical):
#   window = (start, end]  => start excluded, end included
WINDOW_BOUNDARY = "(start, end]"

# -----------------------------
# Inference (pairwise discordant comparisons)
# -----------------------------
# We compute b_i / c_i across *pairs* (case vs each matched control)
# OR_i = (b_i + pc) / (c_i + pc) with CI on log scale
PSEUDOCOUNT = 0.5

# Evidence threshold (report cleanliness + avoids unstable claims)
MIN_DISCORDANT = 6   # (b+c) minimum for an ingredient to be presented prominently

# Multiple testing (exploratory, non-medical)
Q_THRESH = 0.10      # BH-FDR cutoff for "flagged triggers"

# -----------------------------
# Optional Bayesian posterior layer (belief updating)
# -----------------------------
# This DOES NOT replace BH-FDR; it provides an interpretable uncertainty layer.
# Posterior is on p_i = P(case-only | discordant) where null is p_i = 0.5
USE_BAYES_POSTERIOR = True
BETA_PRIOR_A = 0.5   # Jeffreys prior (weak, symmetric)
BETA_PRIOR_B = 0.5
POSTERIOR_TRIGGER_THRESHOLD = 0.80  # optional UI criterion (do not use alone; combine with q + evidence)

# Optional conservative family pooling as a prior (leave-one-out, weak shrinkage)
# Use only if you implement it explicitly and label as "empirical Bayes / shrinkage".
USE_FAMILY_PRIOR_POOLING = False
FAMILY_POOL_LAMBDA = 0.10           # weak pooling (10% of family evidence)
MIN_FAMILY_DISCORDANT_FOR_POOL = 6  # require evidence before pooling activates

# -----------------------------
# Output / audit settings
# -----------------------------
# Save intermediate audit tables (strata windows, coverage diagnostics)
SAVE_AUDIT_TABLES = True
N_AUDIT_PRINT_EXAMPLES = 3          # number of events to print windows for

print(" SafePlate Trigger Detection config loaded")
print("Paths:")
print(" - MEALS_PATH   :", MEALS_PATH)
print(" - SYMPTOMS_PATH:", SYMPTOMS_PATH)
print(" - GROUPS_PATH  :", GROUPS_PATH)
print(" - OUT_DIR      :", OUT_DIR)

print("\nCore parameters:")
print({
    "EPISODE_GAP_HOURS": EPISODE_GAP_HOURS,
    "EVENT_SEVERITY_THRESHOLD": EVENT_SEVERITY_THRESHOLD,
    "USE_USER_PERCENTILE_FALLBACK": USE_USER_PERCENTILE_FALLBACK,
    "USER_PERCENTILE": USER_PERCENTILE,
    "CASE_HOURS": CASE_HOURS,
    "MIN_LAG_HOURS": MIN_LAG_HOURS,
    "CONTROL_LOOKBACK_DAYS": CONTROL_LOOKBACK_DAYS,
    "N_CONTROLS_PER_EVENT": N_CONTROLS_PER_EVENT,
    "MATCH_DAY_OF_WEEK": MATCH_DAY_OF_WEEK,
    "TIME_MATCH_TOLERANCE_MIN": TIME_MATCH_TOLERANCE_MIN,
    "TIME_MATCH_FALLBACK_MIN": TIME_MATCH_FALLBACK_MIN,
    "CONTROL_SYMPTOM_FREE_MODE": CONTROL_SYMPTOM_FREE_MODE,
    "REQUIRE_SYMPTOM_FREE_CONTROLS": REQUIRE_SYMPTOM_FREE_CONTROLS,
    "MIN_CONTROLS_REQUIRED": MIN_CONTROLS_REQUIRED,
    "WINDOW_BOUNDARY": WINDOW_BOUNDARY,
    "PSEUDOCOUNT": PSEUDOCOUNT,
    "MIN_DISCORDANT": MIN_DISCORDANT,
    "Q_THRESH": Q_THRESH,
    "USE_BAYES_POSTERIOR": USE_BAYES_POSTERIOR,
    "BETA_PRIOR_A": BETA_PRIOR_A,
    "BETA_PRIOR_B": BETA_PRIOR_B,
    "USE_FAMILY_PRIOR_POOLING": USE_FAMILY_PRIOR_POOLING,
    "FAMILY_POOL_LAMBDA": FAMILY_POOL_LAMBDA,
    "MIN_FAMILY_DISCORDANT_FOR_POOL": MIN_FAMILY_DISCORDANT_FOR_POOL,
    "SAVE_AUDIT_TABLES": SAVE_AUDIT_TABLES,
    "N_AUDIT_PRINT_EXAMPLES": N_AUDIT_PRINT_EXAMPLES,
})


 SafePlate Trigger Detection config loaded
Paths:
 - MEALS_PATH   : ../data/synthetic/user_meals.csv
 - SYMPTOMS_PATH: ../data/synthetic/user_symptoms.csv
 - GROUPS_PATH  : ../data/processed/ingredient_groups.csv
 - OUT_DIR      : ../data/processed/trigger_detection

Core parameters:
{'EPISODE_GAP_HOURS': 8, 'EVENT_SEVERITY_THRESHOLD': 7, 'USE_USER_PERCENTILE_FALLBACK': False, 'USER_PERCENTILE': 0.75, 'CASE_HOURS': 24, 'MIN_LAG_HOURS': 2, 'CONTROL_LOOKBACK_DAYS': 28, 'N_CONTROLS_PER_EVENT': 4, 'MATCH_DAY_OF_WEEK': True, 'TIME_MATCH_TOLERANCE_MIN': 0, 'TIME_MATCH_FALLBACK_MIN': 60, 'CONTROL_SYMPTOM_FREE_MODE': 'no_episode_onset', 'REQUIRE_SYMPTOM_FREE_CONTROLS': True, 'MIN_CONTROLS_REQUIRED': 1, 'WINDOW_BOUNDARY': '(start, end]', 'PSEUDOCOUNT': 0.5, 'MIN_DISCORDANT': 6, 'Q_THRESH': 0.1, 'USE_BAYES_POSTERIOR': True, 'BETA_PRIOR_A': 0.5, 'BETA_PRIOR_B': 0.5, 'USE_FAMILY_PRIOR_POOLING': False, 'FAMILY_POOL_LAMBDA': 0.1, 'MIN_FAMILY_DISCORDANT_FOR_POOL': 6, 'SAVE_AUDIT_TABLES': True, 'N_AUD

## Cell 2 — Data loading, validation, and normalisation

This cell **loads all input datasets** (meals, symptoms, ingredient–family mapping) and applies **strict validation and cleaning** to ensure the downstream analysis is statistically valid and reproducible.

**Why this step matters:**
- Trigger detection is extremely sensitive to **timestamp errors**, **token inconsistencies**, and **schema drift**.
- Any silent data issue here would invalidate the case–control comparisons later.

**What this cell guarantees:**
- **Fail-fast schema checks:** required columns must exist, or execution stops immediately.
- **Strict timestamp parsing:** all meal and symptom times are valid datetimes.
- **Consistent token normalisation:** ingredient and group names are lowercased, trimmed, and whitespace-normalised so identical ingredients are counted correctly.
- **Binary-exposure correctness:** exact duplicate `(meal_id, meal_time, ingredient)` rows are removed, preventing inflated exposure counts.
- **Data integrity checks:**
  - Each `meal_id` maps to exactly one `meal_time`.
  - Each ingredient maps to a single, unambiguous `group_head`.
- **Ingredient-level granularity preserved:** ingredients are *not* merged into groups; group labels are stored only for later annotation and aggregation.
- **Audit diagnostics:** prints coverage statistics, time range, duplicate removals, and flags any ingredients missing from the group mapping.

**Outcome:**
- Produces clean, validated `meals`, `symptoms`, and `groups_df` tables.
- Builds a reliable `ingredient_to_group` mapping.
- Establishes a trustworthy foundation for episode clustering, window construction, and statistical inference in later cells.


In [69]:
# -----------------------------------------------------------
# Cell 2 — Load + validate inputs (FIXED: user_id aware)
# -----------------------------------------------------------

# -----------------------------
# Safety: required globals from Cell 1
# -----------------------------
for name in ["MEALS_PATH", "SYMPTOMS_PATH", "GROUPS_PATH"]:
    if name not in globals():
        raise ValueError(f"Expected `{name}` from Cell 1. Run Cell 1 first.")

# -----------------------------
# Load CSVs
# -----------------------------
meals_raw = pd.read_csv(MEALS_PATH)
symptoms_raw = pd.read_csv(SYMPTOMS_PATH)
groups_raw = pd.read_csv(GROUPS_PATH)

# -----------------------------
# Required columns (FIXED)
# -----------------------------
REQ_MEALS = {"user_id", "meal_id", "meal_time", "ingredient"}
REQ_SYMPTOMS = {"user_id", "symptom_time", "symptom_severity"}
REQ_GROUPS = {"ingredient", "group_head"}

missing_meals = REQ_MEALS - set(meals_raw.columns)
missing_symptoms = REQ_SYMPTOMS - set(symptoms_raw.columns)
missing_groups = REQ_GROUPS - set(groups_raw.columns)

if missing_meals:
    raise ValueError(f"user_meals.csv missing columns: {sorted(missing_meals)}")
if missing_symptoms:
    raise ValueError(f"user_symptoms.csv missing columns: {sorted(missing_symptoms)}")
if missing_groups:
    raise ValueError(f"ingredient_groups.csv missing columns: {sorted(missing_groups)}")

# -----------------------------
# Token normalisation helpers
# -----------------------------
_ws = re.compile(r"\s+")
def norm_token(x) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    s = _ws.sub(" ", s)
    return s

# -----------------------------
# Parse timestamps (fail fast)
# -----------------------------
meals_raw["meal_time"] = pd.to_datetime(meals_raw["meal_time"], errors="raise")
symptoms_raw["symptom_time"] = pd.to_datetime(symptoms_raw["symptom_time"], errors="raise")

# -----------------------------
# Normalise types / cleanup (FIXED: user_id)
# -----------------------------
meals_raw["user_id"] = meals_raw["user_id"].astype(str).str.strip()
symptoms_raw["user_id"] = symptoms_raw["user_id"].astype(str).str.strip()

meals_raw["meal_id"] = meals_raw["meal_id"].astype(str).str.strip()
meals_raw["ingredient"] = meals_raw["ingredient"].map(norm_token)

groups_raw["ingredient"] = groups_raw["ingredient"].map(norm_token)
groups_raw["group_head"] = groups_raw["group_head"].map(norm_token)

# Keep only needed columns (prevents accidental leakage)
meals = meals_raw[["user_id", "meal_id", "meal_time", "ingredient"]].copy()
symptoms = symptoms_raw[["user_id", "symptom_time", "symptom_severity"]].copy()
groups_df = groups_raw[["ingredient", "group_head"]].copy()

# Drop obvious empties / invalid IDs
meals = meals[
    meals["user_id"].ne("") &
    meals["meal_id"].ne("") &
    meals["ingredient"].ne("")
].copy()

symptoms = symptoms[symptoms["user_id"].ne("")].copy()
groups_df = groups_df[groups_df["ingredient"].ne("") & groups_df["group_head"].ne("")].copy()

# Severity: enforce numeric + non-null
symptoms["symptom_severity"] = pd.to_numeric(symptoms["symptom_severity"], errors="raise")
symptoms = symptoms.dropna(subset=["symptom_severity", "symptom_time"]).copy()

# -----------------------------
# Deduplicate & sanity checks (FIXED: per-user)
# -----------------------------
# Meals can legitimately have multiple rows per (user_id, meal_id, ingredient) in messy logs.
# For binary exposure, duplicates are redundant; remove exact duplicates.
before = len(meals)
meals = meals.drop_duplicates(subset=["user_id", "meal_id", "meal_time", "ingredient"]).copy()
dropped_dupes = before - len(meals)

# Ensure (user_id, meal_id) -> unique meal_time (if not, it’s a data integrity issue)
n_times_per_meal = meals.groupby(["user_id", "meal_id"])["meal_time"].nunique()
bad_meals = n_times_per_meal[n_times_per_meal > 1]
if len(bad_meals) > 0:
    examples = bad_meals.head(8).to_dict()
    raise ValueError(
        "Data integrity error: some (user_id, meal_id) values map to multiple meal_time values. "
        "This matters because later we collapse by (user_id, meal_id) and assume one timestamp. "
        "Fix upstream logging or create a composite key. Examples ((user_id, meal_id) -> n_times): "
        f"{examples}"
    )

# Groups mapping: if an ingredient appears multiple times with conflicting group_head, that’s bad.
dup_map = groups_df.groupby("ingredient")["group_head"].nunique()
conflicts = dup_map[dup_map > 1]
if len(conflicts) > 0:
    examples = conflicts.head(10).to_dict()
    raise ValueError(
        "ingredient_groups.csv has conflicting mappings (same ingredient -> multiple group_head). "
        f"Examples: {examples}"
    )

# If duplicates exist with same mapping, keep one.
groups_df = groups_df.drop_duplicates(subset=["ingredient"]).copy()

# -----------------------------
# Build ingredient -> group mapping
# -----------------------------
ingredient_to_group = dict(zip(groups_df["ingredient"], groups_df["group_head"]))

# -----------------------------
# Audit diagnostics (research/report friendly)
# -----------------------------
n_meal_rows = len(meals)
n_users = meals["user_id"].nunique()
n_meals = meals.groupby(["user_id", "meal_id"]).ngroups
n_ings = meals["ingredient"].nunique()
tmin, tmax = meals["meal_time"].min(), meals["meal_time"].max()

# Mapping coverage
meal_vocab = set(meals["ingredient"].unique())
mapping_keys = set(groups_df["ingredient"].unique())
unmapped_ingredients = sorted(list(meal_vocab - mapping_keys))

print(" Loaded + validated (best-practice, user-aware)")
print(f"Users: {n_users}")
print(f"Meals rows: {n_meal_rows} | unique meals (user_id, meal_id): {n_meals} | unique ingredients: {n_ings}")
print(f"Meal time range: {tmin} → {tmax}")
print(f"Symptoms rows: {len(symptoms)} | severity range: {symptoms['symptom_severity'].min()} → {symptoms['symptom_severity'].max()}")
print(f"Family mappings: {len(ingredient_to_group)}")
print(f"Exact duplicate meal rows removed: {dropped_dupes}")

# Coverage warnings (non-fatal, but important to know)
if len(unmapped_ingredients) > 0:
    print(f" Mapping coverage: {len(unmapped_ingredients)} meal ingredients are NOT in ingredient_groups.csv.")
    print("    These will fallback to their own token as group_head (singleton families) later.")
    print("    Example unmapped:", unmapped_ingredients[:15])
else:
    print(" Mapping coverage: all meal ingredients are present in ingredient_groups.csv")

# Optional extra sanity checks (helpful but non-fatal)
missing_symptom_users = sorted(list(set(symptoms["user_id"].unique()) - set(meals["user_id"].unique())))
if len(missing_symptom_users) > 0:
    print(f" Warning: {len(missing_symptom_users)} user_id values appear in symptoms but not meals. Example:", missing_symptom_users[:10])

# Quick display
display(meals.head(8))
display(symptoms.head(8))
display(groups_df.head(8))


 Loaded + validated (best-practice, user-aware)
Users: 3
Meals rows: 2233 | unique meals (user_id, meal_id): 582 | unique ingredients: 25
Meal time range: 2025-11-15 08:03:00 → 2026-01-20 19:34:00
Symptoms rows: 73 | severity range: 1 → 9
Family mappings: 2698
Exact duplicate meal rows removed: 0
 Mapping coverage: 5 meal ingredients are NOT in ingredient_groups.csv.
    These will fallback to their own token as group_head (singleton families) later.
    Example unmapped: ['beans', 'butter', 'oats', 'onion', 'pepper']


,user_id,meal_id,meal_time,ingredient
0,u1,u1_m1,2025-11-15 08:16:00,carrot
1,u1,u1_m1,2025-11-15 08:16:00,cumin
2,u1,u1_m1,2025-11-15 08:16:00,oats
3,u1,u1_m1,2025-11-15 08:16:00,onion
4,u1,u1_m2,2025-11-15 13:12:00,banana
5,u1,u1_m2,2025-11-15 13:12:00,potato
6,u1,u1_m2,2025-11-15 13:12:00,tomato
7,u1,u1_m3,2025-11-15 19:40:00,beans


,user_id,symptom_time,symptom_severity
0,u1,2025-11-18 13:52:00,4
1,u1,2025-11-25 05:59:00,2
2,u1,2025-11-26 04:52:00,5
3,u1,2025-11-27 11:46:00,3
4,u1,2025-12-02 13:41:00,5
5,u1,2025-12-04 16:17:00,5
6,u1,2025-12-04 16:32:00,5
7,u1,2025-12-08 23:58:00,1


,ingredient,group_head
0,7-up,7-up
1,7-up soda,7-up soda
2,9 unbaked pie shell,9 unbaked pie shell
3,9-inch deep dish pie crust,9-inch deep dish pie crust
4,9-inch pie shell,9-inch pie shell
5,a.1 original sauce,a.1
6,accent,accent
7,achiote,achiote


## Cell 3 — Meal-level representation and fast exposure queries

This cell converts raw ingredient logs into a **meal-level representation** and defines **efficient, audit-safe helpers** for querying ingredient exposure inside time windows.

**Why this step exists:**
- The trigger model treats exposure as **binary within a time window** (an ingredient is either present or absent).
- Raw logs are too granular and inefficient for repeated window queries.
- We need both **speed** (many window lookups) and **correct boundary handling** for causal validity.

**What this cell does:**
- **Collapses ingredient rows into meals:**  
  Each `meal_id` becomes a single row with:
  - one timestamp (`meal_time`)
  - a **set of unique ingredients** (duplicates removed)
  - audit fields showing how many rows and unique ingredients were logged per meal.
- **Sorts meals by time** and builds a NumPy datetime array to support fast slicing.
- **Defines window helpers** based on exact boundary rules:
  - Hazard windows are always interpreted as **(start, end]**  
    (exclude the left boundary, include the right boundary).
  - Efficient index lookup is done using binary search (`searchsorted`), not slow filtering.
- **Provides two core exposure functions:**
  - `ingredients_set_in_window(start, end)` → all ingredients eaten in the window (binary exposure).
  - `meal_count_in_window(start, end)` → number of meals contributing exposure (auditability).

**Why this is best practice:**
- Enforces a **clear causal boundary** (no exposure counted at or after symptom onset).
- Prevents over-counting from repeated ingredient logs.
- Makes later statistical steps fast, reproducible, and easy to explain in a report.
- Includes explicit sanity checks to prove boundary logic is correct.

**Outcome:**
- A clean, time-ordered meal table (`meals_by_meal`).
- Reliable, fast exposure queries that underpin all case–control comparisons later.


In [70]:
# -----------------------------
# Cell 3 — Collapse to meal-level + fast window primitives (USER-AWARE)
# -----------------------------

# -----------------------------
# Safety checks
# -----------------------------
if "meals" not in globals():
    raise ValueError("Expected `meals` from Cell 2. Run Cell 2 first.")

# Cell 2 is user-aware in your run, so we REQUIRE user_id here.
need_cols = {"user_id", "meal_id", "meal_time", "ingredient"}
missing = need_cols - set(meals.columns)
if missing:
    raise ValueError(
        f"`meals` missing required columns: {sorted(missing)}. "
        "This Cell 3 assumes Cell 2 produced a user-aware `meals` with `user_id`."
    )

# Ensure sorted + clean dtypes
meals_local = meals.copy()
meals_local["user_id"] = meals_local["user_id"].astype(str).str.strip()
meals_local["meal_id"] = meals_local["meal_id"].astype(str).str.strip()
meals_local["meal_time"] = pd.to_datetime(meals_local["meal_time"], errors="raise")
meals_local["ingredient"] = meals_local["ingredient"].astype(str).str.strip()

# Drop empties defensively (Cell 2 should already do this, but keep it safe)
meals_local = meals_local[
    meals_local["user_id"].ne("") &
    meals_local["meal_id"].ne("") &
    meals_local["ingredient"].ne("")
].copy()

# -----------------------------
# Integrity check: within each user, a meal_id should map to a single meal_time
# (If you allow same meal_id at multiple times, you must fix logging or use a composite key.)
# -----------------------------
n_times = meals_local.groupby(["user_id", "meal_id"])["meal_time"].nunique()
bad = n_times[n_times > 1]
if len(bad) > 0:
    examples = bad.head(10).to_dict()
    raise ValueError(
        "Data integrity error: some (user_id, meal_id) map to multiple meal_time values. "
        "Fix upstream logging or create a composite key. Examples ((user, meal_id) -> n_times): "
        f"{examples}"
    )

# -----------------------------
# Collapse to meal-level (per USER)
# -----------------------------
meals_by_meal = (
    meals_local.sort_values(["user_id", "meal_time"])
              .groupby(["user_id", "meal_id"], as_index=False)
              .agg(
                  meal_time=("meal_time", "first"),
                  ingredients=("ingredient", lambda s: sorted(set(s))),
                  n_rows=("ingredient", "size"),            # audit: how many ingredient rows were logged for this meal
                  n_unique_ings=("ingredient", "nunique"),  # audit: unique ingredient count (binary exposure uses unique)
              )
)

meals_by_meal = meals_by_meal.sort_values(["user_id", "meal_time"]).reset_index(drop=True)

print("Meals collapsed to meal-level (user-aware)")
print("Meal-level rows:", len(meals_by_meal))
print("Users:", meals_by_meal["user_id"].nunique())
print("Unique (user_id, meal_id):", meals_by_meal[["user_id", "meal_id"]].drop_duplicates().shape[0])
print("Global time range:", meals_by_meal["meal_time"].min(), "→", meals_by_meal["meal_time"].max())
print("Avg unique ingredients per meal:", round(float(meals_by_meal["n_unique_ings"].mean()), 2))

display(meals_by_meal.head(10))

# -----------------------------
# FAST time slicing primitives (per user)
# -----------------------------
# We use numpy searchsorted over sorted meal times, but MUST do it per user.
meals_by_user = {
    uid: df.reset_index(drop=True)
    for uid, df in meals_by_meal.groupby("user_id", sort=False)
}
meal_times_np_by_user = {
    uid: df["meal_time"].to_numpy(dtype="datetime64[ns]")
    for uid, df in meals_by_user.items()
}

def _as_dt64(x) -> np.datetime64:
    return np.datetime64(pd.to_datetime(x))

def window_slice_idx(user_id: str, start, end, closed: str = "right"):
    """
    Return (i0, i1) indices for meals of a given user within a time window using searchsorted.

    closed="right"   -> (start, end]  exclude start, include end   (default; hazard windows)
    closed="left"    -> [start, end)  include start, exclude end
    closed="both"    -> [start, end]  include both
    closed="neither" -> (start, end)  exclude both
    """
    user_id = str(user_id).strip()
    if user_id not in meal_times_np_by_user:
        return 0, 0

    mt = meal_times_np_by_user[user_id]
    if len(mt) == 0:
        return 0, 0

    start64 = _as_dt64(start)
    end64 = _as_dt64(end)

    if end64 < start64:
        return 0, 0

    if closed == "right":
        i0 = int(np.searchsorted(mt, start64, side="right"))
        i1 = int(np.searchsorted(mt, end64,   side="right"))
    elif closed == "left":
        i0 = int(np.searchsorted(mt, start64, side="left"))
        i1 = int(np.searchsorted(mt, end64,   side="left"))
    elif closed == "both":
        i0 = int(np.searchsorted(mt, start64, side="left"))
        i1 = int(np.searchsorted(mt, end64,   side="right"))
    elif closed == "neither":
        i0 = int(np.searchsorted(mt, start64, side="right"))
        i1 = int(np.searchsorted(mt, end64,   side="left"))
    else:
        raise ValueError("closed must be one of: {'right','left','both','neither'}")

    if i1 < i0:
        i1 = i0
    return i0, i1

# -----------------------------
# Exposure helpers (binary, per user)
# -----------------------------
def ingredients_set_in_window(user_id: str, start, end) -> set:
    """
    Return UNION of ingredients for meals for a user in the window (start, end] by default.
    Binary exposure: ingredient appears at least once in the window.
    """
    user_id = str(user_id).strip()
    if user_id not in meals_by_user:
        return set()

    i0, i1 = window_slice_idx(user_id, start, end, closed="right")
    if i1 <= i0:
        return set()

    out = set()
    df = meals_by_user[user_id]
    for lst in df.loc[i0:i1-1, "ingredients"]:
        out.update(lst)
    return out

def meal_count_in_window(user_id: str, start, end) -> int:
    """Count meals for a user in (start, end]."""
    i0, i1 = window_slice_idx(user_id, start, end, closed="right")
    return max(0, i1 - i0)

# -----------------------------
# Sanity check on boundaries (per user)
# -----------------------------
print("\nWindow helper boundary sanity check (per user)")
for uid, df in list(meals_by_user.items())[:3]:  # show up to 3 users
    if len(df) < 2:
        continue
    t0 = df["meal_time"].iloc[0]
    t1 = df["meal_time"].iloc[min(5, len(df)-1)]

    example_start = t0
    example_end = t1

    print(f"\nUser: {uid}")
    print("Example window:", example_start, "→", example_end)
    print("Rule used: (start, end] so meal at start is EXCLUDED, meal at end is INCLUDED.")
    print("Meal count:", meal_count_in_window(uid, example_start, example_end))
    print("Ingredients (first 30):", sorted(list(ingredients_set_in_window(uid, example_start, example_end)))[:30])


Meals collapsed to meal-level (user-aware)
Meal-level rows: 582
Users: 3
Unique (user_id, meal_id): 582
Global time range: 2025-11-15 08:03:00 → 2026-01-20 19:34:00
Avg unique ingredients per meal: 3.84


,user_id,meal_id,meal_time,ingredients,n_rows,n_unique_ings
0,u1,u1_m1,2025-11-15 08:16:00,"[carrot, cumin, oats, onion]",4,4
1,u1,u1_m2,2025-11-15 13:12:00,"[banana, potato, tomato]",3,3
2,u1,u1_m3,2025-11-15 19:40:00,"[beans, cumin, lettuce, pure olive oil]",4,4
3,u1,u1_m4,2025-11-16 08:09:00,"[banana, cumin, pepper, tomato]",4,4
4,u1,u1_m5,2025-11-16 19:33:00,"[oats, onion]",2,2
5,u1,u1_m6,2025-11-17 08:18:00,"[apple, cucumber, egg, oats]",4,4
6,u1,u1_m7,2025-11-17 13:02:00,"[pepper, pure olive oil, tomato]",3,3
7,u1,u1_m8,2025-11-17 19:40:00,"[banana, bread, chicken, oats]",4,4
8,u1,u1_m9,2025-11-18 08:20:00,"[beans, chicken, cumin, honey]",4,4
9,u1,u1_m10,2025-11-18 13:12:00,"[chicken, cucumber, cumin, potato, pure olive ...",5,5



Window helper boundary sanity check (per user)

User: u1
Example window: 2025-11-15 08:16:00 → 2025-11-17 08:18:00
Rule used: (start, end] so meal at start is EXCLUDED, meal at end is INCLUDED.
Meal count: 5
Ingredients (first 30): ['apple', 'banana', 'beans', 'cucumber', 'cumin', 'egg', 'lettuce', 'oats', 'onion', 'pepper', 'potato', 'pure olive oil', 'tomato']

User: u2
Example window: 2025-11-15 08:17:00 → 2025-11-16 19:37:00
Rule used: (start, end] so meal at start is EXCLUDED, meal at end is INCLUDED.
Meal count: 5
Ingredients (first 30): ['apple', 'banana', 'beans', 'chicken', 'honey', 'oats', 'onion', 'pure olive oil', 'rice', 'tomato', 'yogurt']

User: u3
Example window: 2025-11-15 08:03:00 → 2025-11-16 19:39:00
Rule used: (start, end] so meal at start is EXCLUDED, meal at end is INCLUDED.
Meal count: 5
Ingredients (first 30): ['apple', 'banana', 'beans', 'bread', 'cumin', 'honey', 'lettuce', 'oats', 'onion', 'potato', 'rice', 'tomato']


## Cell 4 — Symptom episode clustering and event definition

This cell converts raw symptom logs into **independent clinical events** suitable for case-crossover analysis.

**Why this matters:**
- Symptom trackers often record **multiple entries during a single flare**.
- Treating each log as a separate event would **inflate evidence** and violate the independence assumption of case-crossover studies.
- Clustering logs into episodes ensures each event represents **one flare**, not repeated measurements of the same episode.

**What this cell does:**
- Filters symptom logs to **high-severity entries** (above a fixed threshold).
- **Clusters logs into episodes** if they occur within a defined time gap.
- Defines a single **event onset time** per episode (by default, the first log).
- Produces:
  - `episodes_df`: detailed episode summaries (duration, severity, number of logs).
  - `events`: one clean event time per episode for downstream analysis.
- Adds audit metrics (episode duration, gaps between episodes) to support reporting and validation.

**Outcome:**
- A statistically valid set of **independent event times**, preventing double-counting and making the trigger detection results defensible and reproducible.


In [71]:
# -----------------------------
# Cell 4 — Symptom episode clustering (user-aware, copy-paste ready)
# -----------------------------

# -----------------------------
# Parameters (use globals from Cell 1 if present)
# -----------------------------
EPISODE_GAP_HOURS = int(globals().get("EPISODE_GAP_HOURS", 8))
EVENT_SEVERITY_THRESHOLD = float(globals().get("EVENT_SEVERITY_THRESHOLD", 7))

# Optional: define event time within episode
# "start" = first high-severity log time in episode (default)
# "threshold_cross" = first time sev >= threshold within that episode (same as start after filtering)
EVENT_ONSET_MODE = str(globals().get("EVENT_ONSET_MODE", "start")).strip().lower()
if EVENT_ONSET_MODE not in {"start", "threshold_cross"}:
    raise ValueError("EVENT_ONSET_MODE must be 'start' or 'threshold_cross'.")

gap = pd.Timedelta(hours=EPISODE_GAP_HOURS)
thr = EVENT_SEVERITY_THRESHOLD

# -----------------------------
# Safety checks
# -----------------------------
if "symptoms" not in globals():
    raise ValueError("Expected `symptoms` from Cell 2. Run Cell 2 first.")

need_cols = {"symptom_time", "symptom_severity"}
missing = need_cols - set(symptoms.columns)
if missing:
    raise ValueError(f"`symptoms` missing required columns: {sorted(missing)}")

# If user_id not present, treat as single-user data
HAS_USER = "user_id" in symptoms.columns

sym = symptoms.copy()

if HAS_USER:
    sym["user_id"] = sym["user_id"].astype(str).str.strip()
else:
    sym["user_id"] = "u0"  # single-user fallback

sym["symptom_time"] = pd.to_datetime(sym["symptom_time"], errors="raise")
sym["symptom_severity"] = pd.to_numeric(sym["symptom_severity"], errors="raise")

# Drop NA times or severities (fail-safe; you can choose to raise instead)
sym = sym.dropna(subset=["symptom_time", "symptom_severity"]).copy()

# Sort (by user then time)
sym = sym.sort_values(["user_id", "symptom_time"]).reset_index(drop=True)

# -----------------------------
# Keep only logs that qualify as "event logs" (>= threshold)
# -----------------------------
hi = sym[sym["symptom_severity"] >= thr].copy().reset_index(drop=True)

print("Symptom log precheck (user-aware)")
print("Users:", sym["user_id"].nunique())
print("Total symptom logs:", len(sym))
print(f"High-severity logs (>= {thr}):", len(hi))
print("Episode gap hours:", EPISODE_GAP_HOURS)

# -----------------------------
# Episode clustering per user
# -----------------------------
def _cluster_user_episodes(user_hi: pd.DataFrame) -> list[dict]:
    """
    Cluster high-severity symptom logs into episodes for a single user.
    An episode is a maximal sequence where consecutive logs are <= gap apart.
    """
    if len(user_hi) == 0:
        return []

    user_hi = user_hi.sort_values("symptom_time").reset_index(drop=True)
    u = str(user_hi.loc[0, "user_id"])

    episodes = []
    episode_id = 0

    # Start first episode
    cur_start = user_hi.loc[0, "symptom_time"]
    cur_end = user_hi.loc[0, "symptom_time"]
    cur_times = [user_hi.loc[0, "symptom_time"]]
    cur_sevs = [float(user_hi.loc[0, "symptom_severity"])]

    def _finalize(ep_id, start_time, end_time, times, sevs):
        if EVENT_ONSET_MODE == "start":
            event_time = start_time
        else:
            # First threshold-cross (after filtering, this is the first time in the episode)
            event_time = times[0]

        return {
            "user_id": u,
            "episode_id": int(ep_id),
            "start_time": start_time,
            "end_time": end_time,
            "event_time": event_time,
            "n_logs": int(len(times)),
            "max_severity": float(np.max(sevs)),
            "mean_severity": float(np.mean(sevs)),
            "first_severity": float(sevs[0]),
            "last_severity": float(sevs[-1]),
            "duration_hours": float((end_time - start_time).total_seconds() / 3600.0),
        }

    for i in range(1, len(user_hi)):
        t = user_hi.loc[i, "symptom_time"]
        sev = float(user_hi.loc[i, "symptom_severity"])

        if t - cur_end <= gap:
            # same episode
            cur_end = t
            cur_times.append(t)
            cur_sevs.append(sev)
        else:
            # finalize current episode
            episodes.append(_finalize(episode_id, cur_start, cur_end, cur_times, cur_sevs))

            # start new episode
            episode_id += 1
            cur_start = t
            cur_end = t
            cur_times = [t]
            cur_sevs = [sev]

    # finalize last episode
    episodes.append(_finalize(episode_id, cur_start, cur_end, cur_times, cur_sevs))

    return episodes

if len(hi) == 0:
    episodes_df = pd.DataFrame(columns=[
        "user_id", "episode_id", "start_time", "end_time", "event_time",
        "n_logs", "max_severity", "mean_severity",
        "first_severity", "last_severity",
        "duration_hours", "gap_to_next_episode_hours"
    ])
    events = episodes_df[["user_id", "episode_id", "event_time"]].copy()
    print("No episodes found at this severity threshold. `events` will be empty.")
else:
    all_eps = []
    for uid, user_hi in hi.groupby("user_id", sort=False):
        all_eps.extend(_cluster_user_episodes(user_hi))

    episodes_df = pd.DataFrame(all_eps).sort_values(["user_id", "event_time"]).reset_index(drop=True)

    # -----------------------------
    # Inter-episode gap audit (per user)
    # -----------------------------
    episodes_df["gap_to_next_episode_hours"] = np.nan
    for uid, idx in episodes_df.groupby("user_id").groups.items():
        # idx is an index list for this user's rows (already sorted by event_time)
        sub = episodes_df.loc[idx].sort_values("event_time")
        gaps = (sub["event_time"].shift(-1) - sub["end_time"]).dt.total_seconds() / 3600.0
        episodes_df.loc[sub.index, "gap_to_next_episode_hours"] = gaps.values

    # Events table for case-crossover: one row per episode onset
    events = episodes_df[["user_id", "episode_id", "event_time"]].copy()

    print("\nEpisode clustering complete")
    print("Event onset mode:", EVENT_ONSET_MODE)
    print("Total episodes:", len(episodes_df))
    print("Episodes per user:")
    display(episodes_df.groupby("user_id")["episode_id"].count().rename("n_episodes").reset_index())

    print("Median episode duration (h):", float(np.nanmedian(episodes_df["duration_hours"].values)))
    print("Median logs per episode:", float(np.nanmedian(episodes_df["n_logs"].values)))

    display(episodes_df[[
        "user_id", "episode_id", "event_time", "start_time", "end_time",
        "duration_hours", "n_logs",
        "max_severity", "mean_severity",
        "gap_to_next_episode_hours"
    ]].head(50))

    print("\nEvents table ready (episode onsets):", len(events))
    display(events.head(50))


Symptom log precheck (user-aware)
Users: 3
Total symptom logs: 73
High-severity logs (>= 7.0): 13
Episode gap hours: 8

Episode clustering complete
Event onset mode: start
Total episodes: 9
Episodes per user:


,user_id,n_episodes
0,u1,3
1,u2,3
2,u3,3


Median episode duration (h): 0.0
Median logs per episode: 1.0


,user_id,episode_id,event_time,start_time,end_time,duration_hours,n_logs,max_severity,mean_severity,gap_to_next_episode_hours
0,u1,0,2026-01-02 22:48:00,2026-01-02 22:48:00,2026-01-02 23:48:00,1.0,2,8.0,7.5,166.783333
1,u1,1,2026-01-09 22:35:00,2026-01-09 22:35:00,2026-01-10 02:35:00,4.0,2,8.0,8.0,164.050000
2,u1,2,2026-01-16 22:38:00,2026-01-16 22:38:00,2026-01-17 02:38:00,4.0,2,7.0,7.0,NaN
3,u2,0,2026-01-02 22:44:00,2026-01-02 22:44:00,2026-01-02 22:44:00,0.0,1,7.0,7.0,167.466667
4,u2,1,2026-01-09 22:12:00,2026-01-09 22:12:00,2026-01-09 22:12:00,0.0,1,8.0,8.0,167.966667
5,u2,2,2026-01-16 22:10:00,2026-01-16 22:10:00,2026-01-16 22:10:00,0.0,1,9.0,9.0,NaN
6,u3,0,2026-01-02 22:36:00,2026-01-02 22:36:00,2026-01-03 01:36:00,3.0,2,8.0,8.0,164.933333
7,u3,1,2026-01-09 22:32:00,2026-01-09 22:32:00,2026-01-09 22:32:00,0.0,1,9.0,9.0,168.100000
8,u3,2,2026-01-16 22:38:00,2026-01-16 22:38:00,2026-01-16 22:38:00,0.0,1,7.0,7.0,NaN



Events table ready (episode onsets): 9


,user_id,episode_id,event_time
0,u1,0,2026-01-02 22:48:00
1,u1,1,2026-01-09 22:35:00
2,u1,2,2026-01-16 22:38:00
3,u2,0,2026-01-02 22:44:00
4,u2,1,2026-01-09 22:12:00
5,u2,2,2026-01-16 22:10:00
6,u3,0,2026-01-02 22:36:00
7,u3,1,2026-01-09 22:32:00
8,u3,2,2026-01-16 22:38:00


## Cell 5 — Time-stratified case-crossover strata construction

This cell builds the **core analytical structure** of the trigger detection method:  
for each symptom episode, it creates one **case window** and multiple **matched control windows**.

**What this achieves:**
- Implements a **within-person case-crossover design**, where each event is compared only to the same individual at other times.
- Selects controls that are **time-stratified**:
  - same **time-of-day**
  - same **day-of-week**
  - drawn from a recent lookback period
- This reduces bias from daily routines, weekly patterns, and slow temporal trends.

**Key safeguards included:**
- Enforces a **minimum lag** between meals and symptom onset.
- Excludes control windows that are **not symptom-free** (configurable strictness).
- Prevents control windows from **overlapping any case window** (conservative).
- Drops events with **insufficient valid controls** to avoid biased estimates.
- Uses a **seeded random process** so control selection is reproducible.

**Outputs:**
- `strata_df`: one row per event containing the case window, matched control windows, and rich audit metadata explaining how controls were selected or rejected.

This cell ensures the statistical comparison is **fair, unbiased, and defensible** before any ingredient exposure or inference is performed.


In [72]:
# -----------------------------
# Cell 5 — Build case/control strata (USER-AWARE, copy-paste ready)
# -----------------------------

# -----------------------------
# Parameters (use globals from Cell 1)
# -----------------------------
CASE_HOURS = int(globals().get("CASE_HOURS", 24))
MIN_LAG_HOURS = int(globals().get("MIN_LAG_HOURS", 2))

CONTROL_LOOKBACK_DAYS = int(globals().get("CONTROL_LOOKBACK_DAYS", 28))
N_CONTROLS_PER_EVENT = int(globals().get("N_CONTROLS_PER_EVENT", 4))

# NOTE: bool("False") == True in Python, so we parse robustly.
def _parse_bool(x, default=False) -> bool:
    if x is None:
        return bool(default)
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return bool(int(x))
    s = str(x).strip().lower()
    if s in {"true", "t", "1", "yes", "y", "on"}:
        return True
    if s in {"false", "f", "0", "no", "n", "off"}:
        return False
    return bool(default)

MATCH_DAY_OF_WEEK = _parse_bool(globals().get("MATCH_DAY_OF_WEEK", True), default=True)
TIME_MATCH_TOLERANCE_MIN = int(globals().get("TIME_MATCH_TOLERANCE_MIN", 0))
TIME_MATCH_FALLBACK_MIN = int(globals().get("TIME_MATCH_FALLBACK_MIN", 60))

REQUIRE_SYMPTOM_FREE_CONTROLS = _parse_bool(globals().get("REQUIRE_SYMPTOM_FREE_CONTROLS", True), default=True)
CONTROL_SYMPTOM_FREE_MODE = str(globals().get("CONTROL_SYMPTOM_FREE_MODE", "no_episode_onset")).strip().lower()
if CONTROL_SYMPTOM_FREE_MODE not in {"no_episode_onset", "no_any_symptom"}:
    raise ValueError("CONTROL_SYMPTOM_FREE_MODE must be one of: {'no_episode_onset','no_any_symptom'}")

MIN_CONTROLS_REQUIRED = int(globals().get("MIN_CONTROLS_REQUIRED", 1))

# Deterministic sampling
GLOBAL_SEED = int(globals().get("GLOBAL_SEED", 42))
rng_controls = np.random.default_rng(GLOBAL_SEED)

# -----------------------------
# Safety checks (USER-AWARE)
# -----------------------------
if "events" not in globals():
    raise ValueError("Expected `events` from Cell 4. Run Cell 4 first.")
if "episodes_df" not in globals():
    raise ValueError("Expected `episodes_df` from Cell 4. Run Cell 4 first.")
if "symptoms" not in globals():
    raise ValueError("Expected `symptoms` from Cell 2 (for control eligibility checks).")

# Required columns
# events must include user_id + event_time; we optionally keep episode_id if present
req_events = {"user_id", "event_time"}
missing = req_events - set(events.columns)
if missing:
    raise ValueError(f"`events` missing required columns: {sorted(missing)}")

req_eps = {"user_id", "event_time"}
missing = req_eps - set(episodes_df.columns)
if missing:
    raise ValueError(f"`episodes_df` missing required columns: {sorted(missing)}")

req_sym = {"user_id", "symptom_time", "symptom_severity"}
missing = req_sym - set(symptoms.columns)
if missing:
    raise ValueError(f"`symptoms` missing required columns: {sorted(missing)}")

if "WINDOW_BOUNDARY" not in globals():
    print(" WINDOW_BOUNDARY not found; assuming hazard windows are (start, end] for exposures (as in Cell 3).")

events_local = events.copy()
events_local["event_time"] = pd.to_datetime(events_local["event_time"], errors="raise")
events_local["user_id"] = events_local["user_id"].astype(str).str.strip()
# keep optional episode_id
if "episode_id" in events_local.columns:
    events_local["episode_id"] = events_local["episode_id"].astype(str).str.strip()
events_local = events_local.sort_values(["user_id", "event_time"]).reset_index(drop=True)

episodes_local = episodes_df.copy()
episodes_local["event_time"] = pd.to_datetime(episodes_local["event_time"], errors="raise")
episodes_local["user_id"] = episodes_local["user_id"].astype(str).str.strip()
episodes_local = episodes_local.sort_values(["user_id", "event_time"]).reset_index(drop=True)

symptoms_local = symptoms.copy()
symptoms_local["symptom_time"] = pd.to_datetime(symptoms_local["symptom_time"], errors="raise")
symptoms_local["symptom_severity"] = pd.to_numeric(symptoms_local["symptom_severity"], errors="raise")
symptoms_local["user_id"] = symptoms_local["user_id"].astype(str).str.strip()
symptoms_local = symptoms_local.dropna(subset=["symptom_time", "symptom_severity"]).sort_values(["user_id", "symptom_time"]).reset_index(drop=True)

users_in_events = sorted(events_local["user_id"].unique().tolist())

# Build per-user numpy arrays for fast checks
episode_onsets_np_by_user = {}
symptom_times_np_by_user = {}

for uid, g in episodes_local.groupby("user_id", sort=False):
    episode_onsets_np_by_user[uid] = g["event_time"].to_numpy(dtype="datetime64[ns]")

for uid, g in symptoms_local.groupby("user_id", sort=False):
    symptom_times_np_by_user[uid] = g["symptom_time"].to_numpy(dtype="datetime64[ns]")

# -----------------------------
# Helpers
# -----------------------------
def hazard_window(t_anchor: pd.Timestamp, L_hours: int, e_hours: int):
    """
    Hazard window: (T-L, T-e]
    Return (start, end) where start < t_meal <= end under the '(start, end]' convention.
    """
    start = t_anchor - pd.Timedelta(hours=L_hours)
    end = t_anchor - pd.Timedelta(hours=e_hours)
    return start, end

def minute_of_day(ts: pd.Timestamp) -> int:
    return int(ts.hour) * 60 + int(ts.minute)

def minutes_circular_diff(a: int, b: int) -> int:
    d = abs(a - b)
    return min(d, 1440 - d)

def any_time_in_closed_interval(times_np: np.ndarray, start: pd.Timestamp, end: pd.Timestamp) -> bool:
    """
    Check whether any timestamp in times_np lies in [start, end] (closed interval).
    Uses searchsorted for O(log n).
    """
    if times_np is None or len(times_np) == 0:
        return False
    s = np.datetime64(pd.to_datetime(start))
    e = np.datetime64(pd.to_datetime(end))
    i0 = int(np.searchsorted(times_np, s, side="left"))
    i1 = int(np.searchsorted(times_np, e, side="right"))
    return i1 > i0

def window_is_symptom_free(uid: str, ctrl_start: pd.Timestamp, ctrl_end: pd.Timestamp) -> bool:
    """
    Eligibility criterion for controls (PER USER):
    - no_episode_onset: no episode onset within [ctrl_start, ctrl_end]
    - no_any_symptom: no symptom log time within [ctrl_start, ctrl_end]
    """
    if not REQUIRE_SYMPTOM_FREE_CONTROLS:
        return True

    if CONTROL_SYMPTOM_FREE_MODE == "no_episode_onset":
        return not any_time_in_closed_interval(
            episode_onsets_np_by_user.get(uid, np.array([], dtype="datetime64[ns]")),
            ctrl_start,
            ctrl_end,
        )

    return not any_time_in_closed_interval(
        symptom_times_np_by_user.get(uid, np.array([], dtype="datetime64[ns]")),
        ctrl_start,
        ctrl_end,
    )

def overlaps_any_case_window(ctrl_start: pd.Timestamp, ctrl_end: pd.Timestamp, case_windows_for_user) -> bool:
    """
    Conservative guardrail (PER USER):
    exclude control windows that overlap ANY case hazard window for that same user's events.
    """
    for (cs, ce) in case_windows_for_user:
        # overlap if closed intervals intersect
        if not (ctrl_end < cs or ctrl_start > ce):
            return True
    return False

def candidate_control_end_times(
    t_event: pd.Timestamp,
    lookback_days: int,
    tol_min: int,
    require_same_dow: bool,
):
    """
    Generate candidate control anchor times T' within [T-lookback, T-1day],
    matched on:
      - (optional) day-of-week
      - time-of-day within tol_min (circular minutes)
    """
    t_min = t_event - pd.Timedelta(days=lookback_days)
    t_max = t_event - pd.Timedelta(days=1)  # at least 1 day before event

    target_mod = minute_of_day(t_event)
    target_dow = int(t_event.dayofweek)  # Monday=0

    cands = []
    for d in pd.date_range(t_min.normalize(), t_max.normalize(), freq="D"):
        if require_same_dow and int(d.dayofweek) != target_dow:
            continue

        tc = d + pd.Timedelta(hours=t_event.hour, minutes=t_event.minute, seconds=t_event.second)

        if tc < t_min or tc > t_max:
            continue

        diff = minutes_circular_diff(minute_of_day(tc), target_mod)
        if diff <= tol_min:
            cands.append(tc)

    return cands

# -----------------------------
# Precompute all case windows (PER USER, for overlap exclusion)
# -----------------------------
all_case_windows_by_user = defaultdict(list)
for _, row in events_local.iterrows():
    uid = row["user_id"]
    t_event = row["event_time"]
    cs, ce = hazard_window(t_event, CASE_HOURS, MIN_LAG_HOURS)
    all_case_windows_by_user[uid].append((cs, ce))

# -----------------------------
# Build strata per event (PER USER)
# -----------------------------
strata_rows = []
dropped_events = 0

# stable id within this cell (not the episode_id)
event_counter = 0

for _, ev in events_local.iterrows():
    uid = ev["user_id"]
    t_event = ev["event_time"]
    ep_id = ev["episode_id"] if "episode_id" in events_local.columns else None

    case_start, case_end = hazard_window(t_event, CASE_HOURS, MIN_LAG_HOURS)

    # Candidate generation: strict first
    used_tol = TIME_MATCH_TOLERANCE_MIN
    candidates = candidate_control_end_times(
        t_event=t_event,
        lookback_days=CONTROL_LOOKBACK_DAYS,
        tol_min=TIME_MATCH_TOLERANCE_MIN,
        require_same_dow=MATCH_DAY_OF_WEEK,
    )

    # Fallback if too few
    if len(candidates) < N_CONTROLS_PER_EVENT and TIME_MATCH_FALLBACK_MIN > TIME_MATCH_TOLERANCE_MIN:
        used_tol = TIME_MATCH_FALLBACK_MIN
        candidates = candidate_control_end_times(
            t_event=t_event,
            lookback_days=CONTROL_LOOKBACK_DAYS,
            tol_min=TIME_MATCH_FALLBACK_MIN,
            require_same_dow=MATCH_DAY_OF_WEEK,
        )

    accepted = []
    reject_reasons = defaultdict(int)

    for tc_end in candidates:
        ctrl_start, ctrl_end = hazard_window(tc_end, CASE_HOURS, MIN_LAG_HOURS)

        # Avoid overlap with any case window for THIS user
        if overlaps_any_case_window(ctrl_start, ctrl_end, all_case_windows_by_user.get(uid, [])):
            reject_reasons["overlaps_any_case_window_same_user"] += 1
            continue

        # Symptom-free eligibility (THIS user)
        if not window_is_symptom_free(uid, ctrl_start, ctrl_end):
            reject_reasons[f"not_symptom_free_{CONTROL_SYMPTOM_FREE_MODE}"] += 1
            continue

        accepted.append((ctrl_start, ctrl_end, tc_end))

    # Sample up to K (deterministic)
    if len(accepted) > N_CONTROLS_PER_EVENT:
        idx = rng_controls.choice(len(accepted), size=N_CONTROLS_PER_EVENT, replace=False)
        accepted = [accepted[i] for i in sorted(idx)]

    n_controls = len(accepted)

    row_base = {
        "user_id": uid,
        "event_id": int(event_counter),
        "event_time": t_event,
        "case_start": case_start,
        "case_end": case_end,
        "n_controls": int(n_controls),
        "time_match_tol_min_used": int(used_tol),
        "candidates_total": int(len(candidates)),
        "accepted_total": int(n_controls),
        "lookback_days": int(CONTROL_LOOKBACK_DAYS),
        "min_lag_hours": int(MIN_LAG_HOURS),
        "case_hours": int(CASE_HOURS),
        "match_day_of_week": bool(MATCH_DAY_OF_WEEK),
        "symptom_free_mode": CONTROL_SYMPTOM_FREE_MODE if REQUIRE_SYMPTOM_FREE_CONTROLS else "off",
        "reject_reasons": dict(reject_reasons),
    }
    if ep_id is not None:
        row_base["episode_id"] = ep_id

    if n_controls < MIN_CONTROLS_REQUIRED:
        dropped_events += 1
        strata_rows.append({
            **row_base,
            "control_windows": [],
            "control_end_times": [],
            "dropped": True,
            "drop_reason": f"insufficient_controls(<{MIN_CONTROLS_REQUIRED})",
        })
    else:
        strata_rows.append({
            **row_base,
            "control_windows": [(cs, ce) for (cs, ce, _) in accepted],
            "control_end_times": [tc for (_, _, tc) in accepted],
            "dropped": False,
            "drop_reason": "",
        })

    event_counter += 1

strata_df = pd.DataFrame(strata_rows).sort_values(["user_id", "event_time"]).reset_index(drop=True)

# -----------------------------
# Summary prints (research/report friendly)
# -----------------------------
print("Strata built (user-aware):", len(strata_df))
print("Users:", strata_df["user_id"].nunique())
print("Dropped events (insufficient controls):", dropped_events)

non_dropped = strata_df.loc[~strata_df["dropped"]]
if len(non_dropped) > 0:
    print("Controls per non-dropped event:", list(non_dropped["n_controls"].values))
else:
    print("No non-dropped events (all events dropped). Check your matching + symptom-free constraints.")

display_cols = [
    "user_id", "event_id", "event_time",
    "case_start", "case_end",
    "n_controls", "dropped",
    "time_match_tol_min_used",
    "candidates_total", "accepted_total",
    "symptom_free_mode"
]
if "episode_id" in strata_df.columns:
    display_cols.insert(2, "episode_id")

display(strata_df[display_cols])

# -----------------------------
# Audit printout (robot-proof)
# -----------------------------
n_show = int(globals().get("N_AUDIT_PRINT_EXAMPLES", 3))
print("\n Audit: sample strata windows")
for _, row in strata_df.head(n_show).iterrows():
    tag = f"User {row['user_id']} | Event {int(row['event_id'])}"
    if "episode_id" in row.index and pd.notna(row["episode_id"]):
        tag += f" | Episode {row['episode_id']}"
    print(f"\n{tag} @ {row['event_time']} (dropped={row['dropped']})")
    print("  Case (T-L, T-e] :", row["case_start"], "->", row["case_end"])
    if row["dropped"]:
        print("  DROP reason:", row["drop_reason"])
        print("  Reject reasons:", row["reject_reasons"])
        continue
    for j, (cs, ce) in enumerate(row["control_windows"], start=1):
        print(f"  Control {j}     :", cs, "->", ce)
    if row["reject_reasons"]:
        print("  Rejections:", row["reject_reasons"])

# -----------------------------
# Optional: save audit table
# -----------------------------
if _parse_bool(globals().get("SAVE_AUDIT_TABLES", True), default=True) and "OUT_DIR" in globals():
    strata_path = Path(OUT_DIR) / "audit_strata_windows.csv"
    # Convert complex columns to strings for CSV
    out_df = strata_df.copy()
    out_df["reject_reasons"] = out_df["reject_reasons"].astype(str)
    out_df["control_windows"] = out_df["control_windows"].astype(str)
    out_df["control_end_times"] = out_df["control_end_times"].astype(str)
    out_df.to_csv(strata_path, index=False)
    print("\n Saved audit strata:", strata_path)


Strata built (user-aware): 9
Users: 3
Dropped events (insufficient controls): 0
Controls per non-dropped event: [4, 3, 2, 4, 3, 2, 4, 3, 2]


,user_id,event_id,episode_id,event_time,case_start,case_end,n_controls,dropped,time_match_tol_min_used,candidates_total,accepted_total,symptom_free_mode
0,u1,0,0,2026-01-02 22:48:00,2026-01-01 22:48:00,2026-01-02 20:48:00,4,False,0,4,4,no_episode_onset
1,u1,1,1,2026-01-09 22:35:00,2026-01-08 22:35:00,2026-01-09 20:35:00,3,False,0,4,3,no_episode_onset
2,u1,2,2,2026-01-16 22:38:00,2026-01-15 22:38:00,2026-01-16 20:38:00,2,False,0,4,2,no_episode_onset
3,u2,3,0,2026-01-02 22:44:00,2026-01-01 22:44:00,2026-01-02 20:44:00,4,False,0,4,4,no_episode_onset
4,u2,4,1,2026-01-09 22:12:00,2026-01-08 22:12:00,2026-01-09 20:12:00,3,False,0,4,3,no_episode_onset
5,u2,5,2,2026-01-16 22:10:00,2026-01-15 22:10:00,2026-01-16 20:10:00,2,False,0,4,2,no_episode_onset
6,u3,6,0,2026-01-02 22:36:00,2026-01-01 22:36:00,2026-01-02 20:36:00,4,False,0,4,4,no_episode_onset
7,u3,7,1,2026-01-09 22:32:00,2026-01-08 22:32:00,2026-01-09 20:32:00,3,False,0,4,3,no_episode_onset
8,u3,8,2,2026-01-16 22:38:00,2026-01-15 22:38:00,2026-01-16 20:38:00,2,False,0,4,2,no_episode_onset



 Audit: sample strata windows

User u1 | Event 0 | Episode 0 @ 2026-01-02 22:48:00 (dropped=False)
  Case (T-L, T-e] : 2026-01-01 22:48:00 -> 2026-01-02 20:48:00
  Control 1     : 2025-12-04 22:48:00 -> 2025-12-05 20:48:00
  Control 2     : 2025-12-11 22:48:00 -> 2025-12-12 20:48:00
  Control 3     : 2025-12-18 22:48:00 -> 2025-12-19 20:48:00
  Control 4     : 2025-12-25 22:48:00 -> 2025-12-26 20:48:00

User u1 | Event 1 | Episode 1 @ 2026-01-09 22:35:00 (dropped=False)
  Case (T-L, T-e] : 2026-01-08 22:35:00 -> 2026-01-09 20:35:00
  Control 1     : 2025-12-11 22:35:00 -> 2025-12-12 20:35:00
  Control 2     : 2025-12-18 22:35:00 -> 2025-12-19 20:35:00
  Control 3     : 2025-12-25 22:35:00 -> 2025-12-26 20:35:00
  Rejections: {'overlaps_any_case_window_same_user': 1}

User u1 | Event 2 | Episode 2 @ 2026-01-16 22:38:00 (dropped=False)
  Case (T-L, T-e] : 2026-01-15 22:38:00 -> 2026-01-16 20:38:00
  Control 1     : 2025-12-18 22:38:00 -> 2025-12-19 20:38:00
  Control 2     : 2025-12-25 

## Cell 6 — Per-stratum ingredient exposure construction

This cell translates each **case–control stratum** into the ingredient exposures used for statistical inference.

**What this does:**
- Computes **binary ingredient exposure** for each window:
  - an ingredient is exposed if it appears at least once.
- Enforces the audit-critical window boundary **(start, end]** consistently.
- Builds:
  - a **case exposure set** per event,
  - individual **control exposure sets**,
  - and a **control-union set** used for discordant comparisons (U/V).

**Key safeguards and diagnostics:**
- Skips strata that were **dropped** or have **no valid controls** (prevents artificial inflation).
- Records meal coverage and exposure density to explain weak or missing evidence.
- Flags windows outside meal coverage or with zero meals.
- Provides explicit audit output showing which ingredients contribute to U (case-only) and V (control-only).

**Outcome:**
- A clean, auditable exposure representation for each event, ready for statistically valid trigger inference in the next step.


In [73]:
# ============================================================
# Cell 6 — Build case/control exposure sets per stratum (user-aware)
# ============================================================

from pathlib import Path  # safe even if already imported

# -----------------------------
# Safety checks (user-aware)
# -----------------------------
# FIX: we do NOT require `window_slice_idx` anymore because this cell builds its own user-aware slicer.
required = ["meals_by_meal", "strata_df"]
for name in required:
    if name not in globals():
        raise ValueError(
            f"Expected `{name}` from earlier cells. Ensure Cells 3 and 5 ran successfully."
        )

# We REQUIRE these columns (user-aware)
need_cols = {"user_id", "event_id", "event_time", "case_start", "case_end", "control_windows", "n_controls"}
missing = need_cols - set(strata_df.columns)
if missing:
    raise ValueError(f"`strata_df` missing required columns: {sorted(missing)} (run Cell 5).")

need_meals_cols = {"user_id", "meal_time", "ingredients"}
missing_m = need_meals_cols - set(meals_by_meal.columns)
if missing_m:
    raise ValueError(f"`meals_by_meal` missing required columns: {sorted(missing_m)} (run Cell 3).")

# Boundary string for report clarity
WINDOW_BOUNDARY = str(globals().get("WINDOW_BOUNDARY", "(start, end]")).strip()
if WINDOW_BOUNDARY != "(start, end]":
    # We still proceed because helpers below implement (start,end], but warn for notebook consistency.
    print("WARNING: WINDOW_BOUNDARY global is not '(start, end]'.")
    print("         This notebook ASSUMES (start, end] for exposure (start EXCLUDED, end INCLUDED).")
    print("         Set WINDOW_BOUNDARY='(start, end]' in Cell 1 for consistency.")

# -----------------------------
# Ensure dtypes + sort
# -----------------------------
meals_by_meal_local = meals_by_meal.copy()
meals_by_meal_local["meal_time"] = pd.to_datetime(meals_by_meal_local["meal_time"], errors="raise")
meals_by_meal_local["user_id"] = meals_by_meal_local["user_id"].astype(str).str.strip()
meals_by_meal_local = meals_by_meal_local.sort_values(["user_id", "meal_time"]).reset_index(drop=True)

strata_df_local = strata_df.copy()
for c in ["event_time", "case_start", "case_end"]:
    strata_df_local[c] = pd.to_datetime(strata_df_local[c], errors="raise")
strata_df_local["user_id"] = strata_df_local["user_id"].astype(str).str.strip()
strata_df_local = strata_df_local.sort_values(["user_id", "event_time"]).reset_index(drop=True)

# Ensure control_windows is always a list (robustness)
if "control_windows" in strata_df_local.columns:
    strata_df_local["control_windows"] = strata_df_local["control_windows"].apply(
        lambda x: [] if x is None else x
    )

# -----------------------------
# Precompute per-user fast slicing arrays
# -----------------------------
# Build a dict: user_id -> (times_np, ingredients_lists)
_user_meals = {}
for uid, g in meals_by_meal_local.groupby("user_id", sort=False):
    g = g.sort_values("meal_time").reset_index(drop=True)
    times_np = g["meal_time"].to_numpy(dtype="datetime64[ns]")
    ing_lists = g["ingredients"].tolist()  # each row is list[str]
    _user_meals[uid] = (times_np, ing_lists)

def _as_dt64(x) -> np.datetime64:
    return np.datetime64(pd.to_datetime(x))

def window_slice_idx_user(user_id: str, start, end, closed: str = "right"):
    """
    Return (i0, i1) indices for meals within a time window using np.searchsorted,
    but scoped to a single user.

    closed="right" means (start, end]  -> exclude exactly start, include exactly end
    """
    user_id = str(user_id).strip()
    if user_id not in _user_meals:
        return 0, 0

    meal_times_np = _user_meals[user_id][0]
    if len(meal_times_np) == 0:
        return 0, 0

    start64 = _as_dt64(start)
    end64 = _as_dt64(end)

    if end64 < start64:
        return 0, 0

    if closed == "right":
        i0 = int(np.searchsorted(meal_times_np, start64, side="right"))
        i1 = int(np.searchsorted(meal_times_np, end64,   side="right"))
    elif closed == "left":
        i0 = int(np.searchsorted(meal_times_np, start64, side="left"))
        i1 = int(np.searchsorted(meal_times_np, end64,   side="left"))
    elif closed == "both":
        i0 = int(np.searchsorted(meal_times_np, start64, side="left"))
        i1 = int(np.searchsorted(meal_times_np, end64,   side="right"))
    elif closed == "neither":
        i0 = int(np.searchsorted(meal_times_np, start64, side="right"))
        i1 = int(np.searchsorted(meal_times_np, end64,   side="left"))
    else:
        raise ValueError("closed must be one of: {'right','left','both','neither'}")

    if i1 < i0:
        i1 = i0
    return i0, i1

def ingredients_set_in_window_user(user_id: str, start, end) -> set:
    """
    Return UNION of ingredients for meals of ONE user in (start, end].
    Binary exposure: ingredient appears at least once in the window.
    """
    user_id = str(user_id).strip()
    if user_id not in _user_meals:
        return set()

    _, ing_lists = _user_meals[user_id]
    i0, i1 = window_slice_idx_user(user_id, start, end, closed="right")
    if i1 <= i0:
        return set()

    out = set()
    for lst in ing_lists[i0:i1]:  # i1 is exclusive
        out.update(lst)
    return out

def meal_count_in_window_user(user_id: str, start, end) -> int:
    """Count meals for ONE user in (start, end]."""
    i0, i1 = window_slice_idx_user(user_id, start, end, closed="right")
    return max(0, i1 - i0)

# -----------------------------
# Window coverage diagnostics (per-user)
# -----------------------------
meal_coverage = (
    meals_by_meal_local.groupby("user_id")["meal_time"]
    .agg(meal_min="min", meal_max="max")
    .reset_index()
)
print("Meal coverage per user:")
display(meal_coverage)

# Fast lookup for coverage
_cov = {r["user_id"]: (r["meal_min"], r["meal_max"]) for _, r in meal_coverage.iterrows()}

# -----------------------------
# Build sets per stratum
# -----------------------------
case_sets = []
control_sets_list = []
control_union_sets = []

n_case_meals_list = []
n_control_meals_total_list = []
n_control_meals_per_window_list = []

n_case_unique_ings_list = []
n_ctrl_unique_ings_union_list = []

coverage_flags = []   # list[str] per stratum
skip_flags = []       # bool per stratum

n_skipped_dropped = 0
n_skipped_no_controls = 0

for _, s in strata_df_local.iterrows():
    uid = str(s["user_id"]).strip()
    dropped = bool(s["dropped"]) if "dropped" in strata_df_local.columns else False
    ctrl_windows = s["control_windows"] if isinstance(s["control_windows"], list) else []

    if dropped:
        n_skipped_dropped += 1
        case_sets.append(set())
        control_sets_list.append([])
        control_union_sets.append(set())
        n_case_meals_list.append(0)
        n_control_meals_total_list.append(0)
        n_control_meals_per_window_list.append([])
        n_case_unique_ings_list.append(0)
        n_ctrl_unique_ings_union_list.append(0)
        coverage_flags.append(["dropped"])
        skip_flags.append(True)
        continue

    if len(ctrl_windows) == 0:
        n_skipped_no_controls += 1
        case_sets.append(set())
        control_sets_list.append([])
        control_union_sets.append(set())
        n_case_meals_list.append(0)
        n_control_meals_total_list.append(0)
        n_control_meals_per_window_list.append([])
        n_case_unique_ings_list.append(0)
        n_ctrl_unique_ings_union_list.append(0)
        coverage_flags.append(["no_controls"])
        skip_flags.append(True)
        continue

    cs, ce = s["case_start"], s["case_end"]

    # --- Case window exposure (user-aware)
    case_set = set(ingredients_set_in_window_user(uid, cs, ce))
    n_case_meals = int(meal_count_in_window_user(uid, cs, ce))

    # --- Controls (user-aware)
    ctrl_sets = []
    ctrl_union = set()
    ctrl_meals_each = []
    ctrl_meals_total = 0

    for (w_start, w_end) in ctrl_windows:
        st = set(ingredients_set_in_window_user(uid, w_start, w_end))
        ctrl_sets.append(st)
        ctrl_union |= st

        m = int(meal_count_in_window_user(uid, w_start, w_end))
        ctrl_meals_each.append(m)
        ctrl_meals_total += m

    # --- Coverage flags (audit)
    flags = []
    if uid in _cov:
        meal_min_u, meal_max_u = _cov[uid]

        # outside meal coverage often yields empty windows
        if (ce < meal_min_u) or (cs > meal_max_u):
            flags.append("case_window_outside_user_meal_coverage")

        if any(
            (pd.to_datetime(w_end) < meal_min_u) or (pd.to_datetime(w_start) > meal_max_u)
            for (w_start, w_end) in ctrl_windows
        ):
            flags.append("some_control_window_outside_user_meal_coverage")
    else:
        flags.append("no_meal_coverage_for_user")

    if n_case_meals == 0:
        flags.append("zero_case_meals")
    if ctrl_meals_total == 0:
        flags.append("zero_control_meals")

    # Store
    case_sets.append(case_set)
    control_sets_list.append(ctrl_sets)
    control_union_sets.append(ctrl_union)

    n_case_meals_list.append(n_case_meals)
    n_control_meals_total_list.append(int(ctrl_meals_total))
    n_control_meals_per_window_list.append(ctrl_meals_each)

    n_case_unique_ings_list.append(int(len(case_set)))
    n_ctrl_unique_ings_union_list.append(int(len(ctrl_union)))

    coverage_flags.append(flags)
    skip_flags.append(False)

# -----------------------------
# Attach to strata_df (global)
# -----------------------------
strata_df_local = strata_df_local.copy()
strata_df_local["skip_inference"] = skip_flags
strata_df_local["coverage_flags"] = coverage_flags

strata_df_local["case_set"] = case_sets
strata_df_local["control_sets"] = control_sets_list
strata_df_local["control_union_set"] = control_union_sets

strata_df_local["n_case_meals"] = n_case_meals_list
strata_df_local["n_control_meals_total"] = n_control_meals_total_list
strata_df_local["n_control_meals_per_window"] = n_control_meals_per_window_list

strata_df_local["n_case_unique_ings"] = n_case_unique_ings_list
strata_df_local["n_ctrl_unique_ings_union"] = n_ctrl_unique_ings_union_list

# Replace global strata_df
strata_df = strata_df_local

print("Case/control exposure sets built (user-aware).")
print("Boundary:", "(start, end]  (start EXCLUDED, end INCLUDED)")
print("Skipped strata (dropped):", n_skipped_dropped)
print("Skipped strata (no controls):", n_skipped_no_controls)
print("Usable strata for inference:", int((~strata_df["skip_inference"]).sum()), "/", len(strata_df))

display(
    strata_df[[
        "user_id",
        "event_id",
        "episode_id" if "episode_id" in strata_df.columns else "event_id",
        "event_time",
        "n_case_meals", "n_controls", "n_control_meals_total",
        "n_case_unique_ings", "n_ctrl_unique_ings_union",
        "skip_inference", "coverage_flags",
        "case_set", "control_union_set"
    ]].head(20)
)

# -----------------------------
# Auditability: print discordant contributions (sample)
# -----------------------------
N_AUDIT_PRINT = int(globals().get("N_AUDIT_PRINT_STRATA", 5))
print("\nAudit: discordant ingredients for first usable strata")
shown = 0
for _, s in strata_df.iterrows():
    if bool(s["skip_inference"]):
        continue
    only_case = sorted(list(set(s["case_set"]) - set(s["control_union_set"])))   # increments U
    only_ctrl = sorted(list(set(s["control_union_set"]) - set(s["case_set"])))   # increments V
    print(f"\nUser {s['user_id']} | Event {int(s['event_id'])} @ {s['event_time']}")
    print("  Case meals:", int(s["n_case_meals"]), "| Control meals total:", int(s["n_control_meals_total"]))
    print("  Case-only (U contributions):", only_case[:50])
    print("  Control-only (V contributions):", only_ctrl[:50])
    if s["coverage_flags"]:
        print("  Coverage flags:", s["coverage_flags"])
    shown += 1
    if shown >= N_AUDIT_PRINT:
        break

# -----------------------------
# Optional: save exposure audit
# -----------------------------
if bool(globals().get("SAVE_AUDIT_TABLES", True)) and "OUT_DIR" in globals():
    out_path = Path(OUT_DIR) / "audit_strata_exposures.csv"
    _save = strata_df.copy()
    for col in ["coverage_flags", "case_set", "control_union_set", "control_sets", "n_control_meals_per_window"]:
        if col in _save.columns:
            _save[col] = _save[col].astype(str)
    _save.to_csv(out_path, index=False)
    print("\nSaved exposure audit:", out_path)


Meal coverage per user:


,user_id,meal_min,meal_max
0,u1,2025-11-15 08:16:00,2026-01-20 19:29:00
1,u2,2025-11-15 08:17:00,2026-01-20 19:34:00
2,u3,2025-11-15 08:03:00,2026-01-20 19:28:00


Case/control exposure sets built (user-aware).
Boundary: (start, end]  (start EXCLUDED, end INCLUDED)
Skipped strata (dropped): 0
Skipped strata (no controls): 0
Usable strata for inference: 9 / 9


,user_id,event_id,episode_id,event_time,n_case_meals,n_controls,n_control_meals_total,n_case_unique_ings,n_ctrl_unique_ings_union,skip_inference,coverage_flags,case_set,control_union_set
0,u1,0,0,2026-01-02 22:48:00,3,4,11,8,18,False,[],"{chicken, beans, egg, oats, yogurt, cumin, ban...","{honey, rice, chicken, pepper, cheese, lettuce..."
1,u1,1,1,2026-01-09 22:35:00,3,3,8,13,17,False,[],"{honey, pepper, chicken, beans, egg, lettuce, ...","{honey, rice, chicken, pepper, cheese, lettuce..."
2,u1,2,2,2026-01-16 22:38:00,3,2,5,16,14,False,[],"{honey, butter, pepper, chicken, cheese, oats,...","{tomato, honey, potato, pepper, cheese, chicke..."
3,u2,3,0,2026-01-02 22:44:00,3,4,12,11,19,False,[],"{tomato, honey, potato, rice, cheese, chicken,...","{honey, rice, chicken, pepper, lettuce, oats, ..."
4,u2,4,1,2026-01-09 22:12:00,3,3,9,10,16,False,[],"{tomato, honey, potato, rice, pepper, beans, y...","{honey, pepper, chicken, lettuce, oats, onion,..."
5,u2,5,2,2026-01-16 22:10:00,3,2,6,10,15,False,[],"{honey, pepper, egg, lettuce, pure olive oil, ...","{honey, pepper, chicken, lettuce, oats, onion,..."
6,u3,6,0,2026-01-02 22:36:00,1,4,11,4,19,False,[],"{onion, bread, potato, pure olive oil}","{honey, rice, pepper, chicken, lettuce, oats, ..."
7,u3,7,1,2026-01-09 22:32:00,3,3,8,12,16,False,[],"{potato, pepper, cheese, chicken, beans, lettu...","{honey, rice, pepper, chicken, oats, onion, cr..."
8,u3,8,2,2026-01-16 22:38:00,3,2,5,10,11,False,[],"{tomato, honey, chicken, beans, egg, pure oliv...","{tomato, honey, potato, rice, chicken, oats, p..."



Audit: discordant ingredients for first usable strata

User u1 | Event 0 @ 2026-01-02 22:48:00
  Case meals: 3 | Control meals total: 11
  Case-only (U contributions): ['egg', 'yogurt']
  Control-only (V contributions): ['bread', 'carrot', 'cheese', 'cream', 'honey', 'lettuce', 'onion', 'pepper', 'potato', 'pure olive oil', 'rice', 'tomato']

User u1 | Event 1 @ 2026-01-09 22:35:00
  Case meals: 3 | Control meals total: 8
  Case-only (U contributions): ['cucumber', 'egg', 'milk', 'pasta']
  Control-only (V contributions): ['apple', 'banana', 'bread', 'cheese', 'onion', 'potato', 'rice', 'tomato']

User u1 | Event 2 @ 2026-01-16 22:38:00
  Case meals: 3 | Control meals total: 5
  Case-only (U contributions): ['banana', 'butter', 'milk', 'pasta']
  Control-only (V contributions): ['beans', 'lettuce']

User u2 | Event 3 @ 2026-01-02 22:44:00
  Case meals: 3 | Control meals total: 12
  Case-only (U contributions): ['cheese']
  Control-only (V contributions): ['apple', 'carrot', 'cucumber'

## Cell 7 — Ingredient-level inference from case–crossover strata

This cell performs the **core statistical inference** step to identify candidate trigger ingredients from the case–crossover strata constructed earlier.

### What is computed per ingredient
For each ingredient, inference is based on **discordant strata only**, using the exposure sets built in Cell 6:

- **U (case-only)**: number of strata in which the ingredient appears in the **case exposure set** but not in the **control-union exposure set**.
- **V (control-only)**: number of strata in which the ingredient appears in the **control-union exposure set** but not in the **case exposure set**.
- **Discordant total**: \( U + V \).

An **effect size** is reported as a stabilised odds ratio:
\[
\text{OR} = \frac{U + \text{pseudocount}}{V + \text{pseudocount}}
\]
which remains well-defined even when \( U = 0 \) or \( V = 0 \).

> **Note:**  
> Discordance is defined using the **union of matched control windows** per stratum, not pairwise comparisons against individual control windows.

### Statistical testing
- Performs an **exact two-sided binomial test** on discordant strata:
  - Tests the null hypothesis  
    \( P(\text{case-only}) = 0.5 \) among discordant observations.
  - Equivalently, this is the exact paired-data analogue of **McNemar’s test** for matched case–control designs.
- Exact inference is used (no asymptotic approximation), which is appropriate for sparse diary data.

### Multiple testing control
- Applies **Benjamini–Hochberg false discovery rate (FDR)** correction across all ingredients.
- Produces **q-values** that control the expected proportion of false discoveries.

### Optional Bayesian layer (interpretability only)
- Computes a **Beta posterior** for  
  \[
  p = P(\text{case-only} \mid \text{discordant})
  \]
  using a weak symmetric prior (Jeffreys: \( \text{Beta}(0.5, 0.5) \)).
- Reports:
  - posterior mean,
  - 95% credible interval,
  - \( P(p > 0.5) \).
- This Bayesian summary is **supplementary** and does not replace frequentist testing or FDR control.

### Quality safeguards
- Excludes strata marked as invalid (dropped or no valid controls).
- Flags **low-evidence ingredients** with insufficient discordant strata.
- Produces both:
  - a complete results table (all ingredients), and
  - a filtered, report-ready table (minimum evidence threshold applied).

### Outcome
- A ranked, statistically controlled list of candidate trigger ingredients,
  annotated with effect size, uncertainty, and evidence strength,
  suitable for downstream evaluation and reporting.


In [74]:
# -----------------------------
# Parameters (use globals from Cell 1)
# -----------------------------
PSEUDOCOUNT = float(globals().get("PSEUDOCOUNT", 0.5))
MIN_DISCORDANT = int(globals().get("MIN_DISCORDANT", 3))

USE_BAYES_POSTERIOR = bool(globals().get("USE_BAYES_POSTERIOR", True))
# Weak symmetric prior (Jeffreys): Beta(0.5, 0.5). Matches your blueprint and is defensible.
BETA_PRIOR_A = float(globals().get("BETA_PRIOR_A", 0.5))
BETA_PRIOR_B = float(globals().get("BETA_PRIOR_B", 0.5))

# If you want a more conservative “flag” threshold later:
Q_THRESH = float(globals().get("Q_THRESH", 0.10))

# Monte Carlo posterior controls
POSTERIOR_MC_SEED = int(globals().get("POSTERIOR_MC_SEED", 123))
POSTERIOR_MC_SAMPLES = int(globals().get("POSTERIOR_MC_SAMPLES", 20000))

# -----------------------------
# Safety checks
# -----------------------------
# FIX: ensure required imports exist in this cell (robust in notebooks)
import numpy as np
import pandas as pd
from collections import defaultdict
from math import comb

required_cols = {"case_set", "control_union_set", "skip_inference"}
missing = required_cols - set(strata_df.columns)
if missing:
    raise ValueError(f"`strata_df` missing required columns: {sorted(missing)} (run Cell 6).")

usable = strata_df[~strata_df["skip_inference"]].copy()
n_strata_total = len(strata_df)
n_strata_usable = len(usable)

print("Strata for inference:", n_strata_usable, "/", n_strata_total)
if n_strata_usable == 0:
    raise ValueError("No usable strata available (all dropped or no controls). Check Cell 5/6 diagnostics.")

# Warn if control meals are globally empty (common pipeline failure mode)
if ("n_control_meals_total" in usable.columns) and (usable["n_control_meals_total"] == 0).all():
    print("WARNING: All usable strata have 0 meals in control windows.")
    print("This usually means your control windows fall outside meal coverage.")
    print("Inference will be meaningless until controls contain meals.\n")

# -----------------------------
# Discordant counting across strata (U/V)
# -----------------------------
U = defaultdict(int)
V = defaultdict(int)

# Also track how many strata had the ingredient *eligible* to be discordant (optional)
# (Here, "eligible" just means it appeared in case or control union. Useful for audit.)
eligible = defaultdict(int)

for _, s in usable.iterrows():
    # Robustly coerce to sets
    cs = s["case_set"]
    cu = s["control_union_set"]

    case_set = set(cs) if isinstance(cs, (set, list, tuple)) else set()
    ctrl_set = set(cu) if isinstance(cu, (set, list, tuple)) else set()

    # ingredients that could matter in this stratum:
    for ing in (case_set | ctrl_set):
        eligible[ing] += 1

    # Discordant contributions:
    for ing in (case_set - ctrl_set):
        U[ing] += 1
    for ing in (ctrl_set - case_set):
        V[ing] += 1

all_ings = sorted(set(U.keys()) | set(V.keys()))
if len(all_ings) == 0:
    raise ValueError(
        "No discordant ingredients found. This can happen if case/control windows are identical "
        "or if meal coverage is missing. Check Cell 6 audit."
    )

print("Ingredients with any discordant evidence:", len(all_ings))

# -----------------------------
# Exact two-sided binomial p-value (McNemar equivalent for paired discordant outcomes)
# H0: P(case-only) = 0.5 among discordant strata
# p = 2 * min( P(X<=u), P(X>=u) ) for X~Bin(n=u+v, 0.5)
# -----------------------------
def binom_cdf_leq(k: int, n: int, p: float) -> float:
    """
    Exact sum_{i=0..k} C(n,i) p^i (1-p)^(n-i).
    n is typically small/moderate in diary settings, so exact is feasible.
    """
    if k < 0:
        return 0.0
    if k >= n:
        return 1.0
    q = 1.0 - p
    s = 0.0
    for i in range(0, k + 1):
        s += comb(n, i) * (p ** i) * (q ** (n - i))
    return float(s)

def two_sided_binom_p(u: int, n: int, p0: float = 0.5) -> float:
    if n <= 0:
        return 1.0
    cdf = binom_cdf_leq(u, n, p0)                  # P(X <= u)
    sf = 1.0 - binom_cdf_leq(u - 1, n, p0)         # P(X >= u)
    p_two = 2.0 * min(cdf, sf)
    return float(min(1.0, max(0.0, p_two)))

# -----------------------------
# BH-FDR (Benjamini–Hochberg)
# -----------------------------
def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]

    q_ranked = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        prev = min(prev, val)
        q_ranked[i] = prev

    qvals = np.empty(n, dtype=float)
    qvals[order] = q_ranked
    return qvals

# -----------------------------
# Optional Bayesian posterior on p_i = P(case-only | discordant)
# Using Beta(a,b) prior -> Beta(a+u, b+v) posterior
#
# Implementation note:
# - We avoid SciPy.
# - We compute P(p>0.5) and 95% CI using Monte Carlo from numpy's Beta sampler.
# - IMPORTANT FIX: use a per-call RNG seeded deterministically so results are reproducible
#   and do not depend on loop order / how many ingredients are processed.
# -----------------------------
def beta_posterior_mc(u: int, v: int, a0: float, b0: float,
                      n_samples: int = 20000, seed: int = 123, salt: int = 0):
    a = a0 + u
    b = b0 + v
    # derive a stable seed per ingredient (salt is provided by caller)
    rng = np.random.default_rng(seed + int(salt))
    draws = rng.beta(a, b, size=int(n_samples))
    p_gt_half = float(np.mean(draws > 0.5))
    mean = float(np.mean(draws))
    lo = float(np.quantile(draws, 0.025))
    hi = float(np.quantile(draws, 0.975))
    return mean, lo, hi, p_gt_half

# -----------------------------
# Build per-ingredient stats table
# -----------------------------
rows = []
PC = float(PSEUDOCOUNT)

# Stable mapping ingredient -> integer salt for reproducible MC per ingredient
# (Avoid Python's built-in hash because it changes across sessions unless PYTHONHASHSEED is fixed.)
def _stable_salt_from_string(s: str) -> int:
    # simple deterministic rolling hash
    h = 0
    for ch in str(s):
        h = (h * 131 + ord(ch)) % 1_000_000_007
    return int(h % 10_000_000)

for ing in all_ings:
    u = int(U.get(ing, 0))
    v = int(V.get(ing, 0))
    n = u + v

    # Effect size: odds ratio with pseudocount (avoids div-by-zero)
    or_hat = float((u + PC) / (v + PC))

    # Exact p-value (paired discordant test)
    pval = two_sided_binom_p(u=u, n=n, p0=0.5)

    row = {
        "ingredient": ing,
        "U_case_not_ctrl": u,
        "V_ctrl_not_case": v,
        "discordant_total": n,
        "eligible_strata": int(eligible.get(ing, 0)),
        "OR": or_hat,
        "p_value": float(pval),
    }

    if USE_BAYES_POSTERIOR:
        salt = _stable_salt_from_string(ing)
        mean, lo, hi, p_gt_half = beta_posterior_mc(
            u=u, v=v,
            a0=BETA_PRIOR_A, b0=BETA_PRIOR_B,
            n_samples=POSTERIOR_MC_SAMPLES,
            seed=POSTERIOR_MC_SEED,
            salt=salt
        )
        row.update({
            "p_case_only_posterior_mean": mean,
            "p_case_only_CI95_low": lo,
            "p_case_only_CI95_high": hi,
            "P_trigger_p_gt_0p5": p_gt_half,
        })

    # Evidence label (for UI/report clarity)
    row["evidence_label"] = "low_evidence" if n < MIN_DISCORDANT else "ok"

    rows.append(row)

ingredient_stats = pd.DataFrame(rows)

# Apply BH-FDR on ingredient-level p-values
ingredient_stats["q_value_bh"] = bh_fdr(ingredient_stats["p_value"].values)

# Rank: prioritize strong signal (OR high), good q (low), and enough discordant evidence
# (This is ranking; your “final triggers” filter happens later.)
ingredient_stats = ingredient_stats.sort_values(
    ["q_value_bh", "p_value", "discordant_total", "OR"],
    ascending=[True, True, False, False]
).reset_index(drop=True)

print("Ingredient-level stats built.")
print(
    "PSEUDOCOUNT:", PC,
    "| MIN_DISCORDANT:", MIN_DISCORDANT,
    "| Posterior:", USE_BAYES_POSTERIOR,
    f"(MC={POSTERIOR_MC_SAMPLES}, seed={POSTERIOR_MC_SEED})"
)
display(ingredient_stats.head(30))

# Evidence-filtered view (clean UI / report-ready)
filtered_ingredient_stats = ingredient_stats[
    ingredient_stats["discordant_total"] >= MIN_DISCORDANT
].copy()

print(f"Filtered ingredients (discordant_total >= {MIN_DISCORDANT}): {len(filtered_ingredient_stats)}")
display(filtered_ingredient_stats.head(30))

# Optional: Save outputs
if bool(globals().get("SAVE_RESULTS", True)) and "OUT_DIR" in globals():
    from pathlib import Path
    out_dir = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    p1 = out_dir / "ingredient_stats_all.csv"
    p2 = out_dir / "ingredient_stats_filtered.csv"
    ingredient_stats.to_csv(p1, index=False)
    filtered_ingredient_stats.to_csv(p2, index=False)
    print("\nSaved:", p1)
    print("Saved:", p2)

# Optional: produce “final triggers” list here (strictly ingredient-level)
final_triggers = filtered_ingredient_stats[
    (filtered_ingredient_stats["q_value_bh"] <= Q_THRESH)
].copy()

print(f"\nFinal triggers (q <= {Q_THRESH} AND discordant_total >= {MIN_DISCORDANT}): {len(final_triggers)}")
display(final_triggers.head(30))


Strata for inference: 9 / 9
Ingredients with any discordant evidence: 25
Ingredient-level stats built.
PSEUDOCOUNT: 0.5 | MIN_DISCORDANT: 6 | Posterior: True (MC=20000, seed=321)


,ingredient,U_case_not_ctrl,V_ctrl_not_case,discordant_total,eligible_strata,OR,p_value,p_case_only_posterior_mean,p_case_only_CI95_low,p_case_only_CI95_high,P_trigger_p_gt_0p5,evidence_label,q_value_bh
0,tomato,0,5,5,9,0.090909,0.06250,0.083671,0.000098,0.381765,0.00605,low_evidence,0.520833
1,apple,0,4,4,9,0.111111,0.12500,0.100149,0.000110,0.441977,0.01475,low_evidence,0.520833
2,carrot,0,4,4,6,0.111111,0.12500,0.099803,0.000139,0.437566,0.01485,low_evidence,0.520833
3,oats,0,4,4,9,0.111111,0.12500,0.100397,0.000115,0.440945,0.01480,low_evidence,0.520833
4,lettuce,1,5,6,8,0.272727,0.21875,0.214171,0.018847,0.553405,0.04520,ok,0.520833
5,rice,1,5,6,7,0.272727,0.21875,0.215309,0.019142,0.562288,0.05060,ok,0.520833
6,chicken,0,3,3,9,0.142857,0.25000,0.124924,0.000150,0.531731,0.03305,low_evidence,0.520833
7,cream,0,3,3,4,0.142857,0.25000,0.124830,0.000142,0.533697,0.03220,low_evidence,0.520833
8,honey,0,3,3,9,0.142857,0.25000,0.125040,0.000159,0.539255,0.03415,low_evidence,0.520833
9,onion,0,3,3,9,0.142857,0.25000,0.125745,0.000139,0.539762,0.03415,low_evidence,0.520833


Filtered ingredients (discordant_total >= 6): 4


,ingredient,U_case_not_ctrl,V_ctrl_not_case,discordant_total,eligible_strata,OR,p_value,p_case_only_posterior_mean,p_case_only_CI95_low,p_case_only_CI95_high,P_trigger_p_gt_0p5,evidence_label,q_value_bh
4,lettuce,1,5,6,8,0.272727,0.21875,0.214171,0.018847,0.553405,0.04520,ok,0.520833
5,rice,1,5,6,7,0.272727,0.21875,0.215309,0.019142,0.562288,0.05060,ok,0.520833
17,cucumber,2,4,6,7,0.555556,0.68750,0.356406,0.076644,0.715196,0.20790,ok,0.954861
18,egg,3,4,7,8,0.777778,1.00000,0.437414,0.137297,0.764436,0.35485,ok,1.000000



Saved: ../data/processed/trigger_detection/ingredient_stats_all.csv
Saved: ../data/processed/trigger_detection/ingredient_stats_filtered.csv

Final triggers (q <= 0.1 AND discordant_total >= 6): 0


,ingredient,U_case_not_ctrl,V_ctrl_not_case,discordant_total,eligible_strata,OR,p_value,p_case_only_posterior_mean,p_case_only_CI95_low,p_case_only_CI95_high,P_trigger_p_gt_0p5,evidence_label,q_value_bh


## Cell 8 — Hierarchical group pooling (ingredient-level, leave-one-out)

This cell adds a **conservative hierarchical Bayesian layer** that stabilizes
ingredient-level inference using **ingredient families**, without collapsing
ingredients.

**Core idea**
- Model each ingredient’s trigger tendency as  
  \( p_i = P(\text{case-only} \mid \text{discordant}) \), with null \( p_i = 0.5 \).
- Start from a weak base prior (Jeffreys Beta).
- **Borrow limited strength from the ingredient’s group** via a
  **leave-one-out (LOO) group prior**, so an ingredient is never informed by its
  own evidence twice.

**How pooling works**
- Compute group-level discordant counts \(U_g, V_g\).
- For ingredient *i* in group *g*, form a prior  
  \( \text{Beta}(a_0 + \lambda U_{g\setminus i},\; b_0 + \lambda V_{g\setminus i}) \).
- Update with ingredient counts \(U_i, V_i\) to obtain a **true posterior**.
- \(\lambda\) is small to ensure conservative shrinkage and avoid over-sharing.

**Outputs**
- **Group diagnostics**: group-level OR, p/q-values, and posterior summaries.
- **Ingredient-enriched table**:
  - pooled posterior \(P(p_i > 0.5)\) with 95% credible interval,
  - classic OR and ingredient-level q-values (unchanged from Cell 7),
  - evidence strength labels and conservative flags.
- **Final triggers** selected using *both* statistical control (q-values) and
  posterior confidence.

**Why this matters**
- Reduces noise for rare ingredients.
- Preserves ingredient granularity.
- Adds an academically defensible, interpretable uncertainty layer suitable for
  dissertation-level work.


In [75]:
# -----------------------------
# Sanity checks
# -----------------------------
if "ingredient_stats" not in globals():
    raise ValueError("Expected `ingredient_stats` from Cell 7.")
if "ingredient_to_group" not in globals() or not isinstance(ingredient_to_group, dict):
    raise ValueError("Expected `ingredient_to_group` dict from Cell 2 (ingredient -> group_head).")
if "bh_fdr" not in globals():
    raise ValueError("Expected `bh_fdr` function from Cell 7.")

need = {"ingredient", "U_case_not_ctrl", "V_ctrl_not_case", "discordant_total", "OR", "p_value", "q_value_bh"}
missing = need - set(ingredient_stats.columns)
if missing:
    raise ValueError(f"`ingredient_stats` missing columns: {sorted(missing)}")

# -----------------------------
# Parameters (use globals from Cell 1/7 when present)
# -----------------------------
PSEUDOCOUNT = float(globals().get("PSEUDOCOUNT", 0.5))
MIN_DISCORDANT = int(globals().get("MIN_DISCORDANT", 3))
Q_THRESH = float(globals().get("Q_THRESH", 0.10))

# Bayesian base prior (Jeffreys by default; conservative for small n)
BETA_PRIOR_A = float(globals().get("BETA_PRIOR_A", 0.5))
BETA_PRIOR_B = float(globals().get("BETA_PRIOR_B", 0.5))

# Group pooling strength (shrinkage). Keep small for dissertation-safety.
# λ=0.10 means: group contributes like 10% of its discordant evidence.
POOL_LAMBDA = float(globals().get("POOL_LAMBDA", 0.10))

# Don't allow group pooling to act if LOO discordant evidence is tiny
MIN_GROUP_LOO_DISCORDANT = int(globals().get("MIN_GROUP_LOO_DISCORDANT", 6))

# Monte Carlo settings (fast; control with globals if needed)
POSTERIOR_MC_SEED = int(globals().get("POSTERIOR_MC_SEED", 321))
POSTERIOR_MC_SAMPLES = int(globals().get("POSTERIOR_MC_SAMPLES", 20000))
rng = np.random.default_rng(POSTERIOR_MC_SEED)

PC = float(PSEUDOCOUNT)

# -----------------------------
# Helper: Beta posterior summaries via MC
# -----------------------------
def beta_mc_summary(a: float, b: float, n: int = 20000):
    draws = rng.beta(a, b, size=int(n))
    return {
        "post_mean": float(np.mean(draws)),
        "CI95_low": float(np.quantile(draws, 0.025)),
        "CI95_high": float(np.quantile(draws, 0.975)),
        "P_gt_0p5": float(np.mean(draws > 0.5)),
    }

# -----------------------------
# Attach group_head (fallback: ingredient itself)
# -----------------------------
ing = ingredient_stats.copy()
ing["ingredient"] = ing["ingredient"].astype(str).str.strip().str.lower()

ingredient_to_group_norm = {str(k).strip().lower(): str(v).strip().lower()
                            for k, v in ingredient_to_group.items()}
ing["group_head"] = ing["ingredient"].map(lambda x: ingredient_to_group_norm.get(x, x))

# -----------------------------
# Group-level aggregation (diagnostic/report)
# -----------------------------
group_uv = (
    ing.groupby("group_head", as_index=False)
      .agg(
          group_U=("U_case_not_ctrl", "sum"),
          group_V=("V_ctrl_not_case", "sum"),
          group_discordant=("discordant_total", "sum"),
          group_n_members=("ingredient", "nunique"),
      )
)

# Group effect size + p/q (p via exact binomial on U out of U+V, same null 0.5)
# (We reuse the exact binomial logic indirectly by recomputing p using a stable helper.)
from math import comb

def binom_cdf_leq(k: int, n: int, p: float) -> float:
    q = 1.0 - p
    s = 0.0
    for i in range(0, k + 1):
        s += comb(n, i) * (p ** i) * (q ** (n - i))
    return float(s)

def two_sided_binom_p(u: int, n: int, p0: float = 0.5) -> float:
    if n <= 0:
        return 1.0
    cdf = binom_cdf_leq(u, n, p0)
    sf = 1.0 - binom_cdf_leq(u - 1, n, p0)
    return float(min(1.0, 2.0 * min(cdf, sf)))

g_rows = []
for _, r in group_uv.iterrows():
    u = int(r["group_U"])
    v = int(r["group_V"])
    n = u + v

    g_or = float((u + PC) / (v + PC))
    g_p = two_sided_binom_p(u=u, n=n, p0=0.5)

    # group posterior without pooling (pure group evidence)
    summ = beta_mc_summary(BETA_PRIOR_A + u, BETA_PRIOR_B + v, n=POSTERIOR_MC_SAMPLES)

    g_rows.append({
        "group_head": r["group_head"],
        "group_U": u,
        "group_V": v,
        "group_discordant": int(r["group_discordant"]),
        "group_n_members": int(r["group_n_members"]),
        "group_OR": g_or,
        "group_p_value": float(g_p),
        "group_post_mean_p": summ["post_mean"],
        "group_CI95_low_p": summ["CI95_low"],
        "group_CI95_high_p": summ["CI95_high"],
        "group_P_gt_0p5": summ["P_gt_0p5"],
    })

group_stats = pd.DataFrame(g_rows)
group_stats["group_q_value_bh"] = bh_fdr(group_stats["group_p_value"].values)

# Rank groups for reporting
group_stats = group_stats.sort_values(
    ["group_P_gt_0p5", "group_q_value_bh", "group_discordant"],
    ascending=[False, True, False]
).reset_index(drop=True)

print(" Group-level diagnostics built (for reporting/interpretation):")
display(group_stats.head(25))

# Build lookup for LOO pooling
group_lookup = group_stats.set_index("group_head")[["group_U", "group_V", "group_discordant", "group_n_members"]].to_dict("index")

# -----------------------------
# Ingredient-level pooled posterior (LOO group prior)
# -----------------------------
pool_post_mean = []
pool_ci_lo = []
pool_ci_hi = []
pool_p_gt = []
loo_u_list = []
loo_v_list = []
loo_d_list = []
prior_a_list = []
prior_b_list = []

for _, r in ing.iterrows():
    g = r["group_head"]
    u_i = int(r["U_case_not_ctrl"])
    v_i = int(r["V_ctrl_not_case"])

    gU = int(group_lookup.get(g, {}).get("group_U", 0))
    gV = int(group_lookup.get(g, {}).get("group_V", 0))

    loo_u = max(0, gU - u_i)
    loo_v = max(0, gV - v_i)
    loo_d = loo_u + loo_v

    # If not enough LOO evidence, revert to base prior
    if loo_d < MIN_GROUP_LOO_DISCORDANT or POOL_LAMBDA <= 0:
        prior_a = BETA_PRIOR_A
        prior_b = BETA_PRIOR_B
    else:
        prior_a = BETA_PRIOR_A + POOL_LAMBDA * loo_u
        prior_b = BETA_PRIOR_B + POOL_LAMBDA * loo_v

    # Posterior update with ingredient evidence
    post_a = prior_a + u_i
    post_b = prior_b + v_i

    summ = beta_mc_summary(post_a, post_b, n=POSTERIOR_MC_SAMPLES)

    loo_u_list.append(int(loo_u))
    loo_v_list.append(int(loo_v))
    loo_d_list.append(int(loo_d))
    prior_a_list.append(float(prior_a))
    prior_b_list.append(float(prior_b))
    pool_post_mean.append(summ["post_mean"])
    pool_ci_lo.append(summ["CI95_low"])
    pool_ci_hi.append(summ["CI95_high"])
    pool_p_gt.append(summ["P_gt_0p5"])

ingredient_enriched = ing.copy()
ingredient_enriched["loo_group_U"] = loo_u_list
ingredient_enriched["loo_group_V"] = loo_v_list
ingredient_enriched["loo_group_discordant"] = loo_d_list
ingredient_enriched["prior_a_used"] = prior_a_list
ingredient_enriched["prior_b_used"] = prior_b_list

ingredient_enriched["pooled_post_mean_p"] = pool_post_mean
ingredient_enriched["pooled_CI95_low_p"] = pool_ci_lo
ingredient_enriched["pooled_CI95_high_p"] = pool_ci_hi
ingredient_enriched["P_trigger_p_gt_0p5_pooled"] = pool_p_gt

# A single "score" for ranking (posterior probability is the cleanest)
ingredient_enriched["trigger_score"] = ingredient_enriched["P_trigger_p_gt_0p5_pooled"]

# Evidence label upgrade (helps UX + report)
def evidence_label(row):
    n = int(row["discordant_total"])
    if n < MIN_DISCORDANT:
        return "low_evidence"
    if n < 6:
        return "moderate_evidence"
    return "strong_evidence"

ingredient_enriched["evidence_label"] = ingredient_enriched.apply(evidence_label, axis=1)

# Conservative final ranking:
#   1) posterior P(trigger) desc
#   2) q asc
#   3) discordant desc
ingredient_enriched = ingredient_enriched.sort_values(
    ["trigger_score", "q_value_bh", "discordant_total", "OR"],
    ascending=[False, True, False, False]
).reset_index(drop=True)
ingredient_enriched["rank"] = np.arange(1, len(ingredient_enriched) + 1)

print("Ingredient-level pooled results built (LOO hierarchical pooling):")
display(ingredient_enriched.head(30))

# -----------------------------
# Final trigger set (ingredient-level inference + evidence gate)
# -----------------------------
# Important: still respect multiple testing at INGREDIENT level (q_value_bh)
# Pooling is for stabilizing probability/interpretation, not for changing q-values.
final_triggers = ingredient_enriched[
    (ingredient_enriched["discordant_total"] >= MIN_DISCORDANT) &
    (ingredient_enriched["q_value_bh"] <= Q_THRESH) &
    (ingredient_enriched["P_trigger_p_gt_0p5_pooled"] >= 0.80)  # conservative posterior threshold
].copy()

print(f"Final triggers (q<= {Q_THRESH}, discordant>= {MIN_DISCORDANT}, P_trigger>=0.80): {len(final_triggers)}")
display(final_triggers[[
    "rank", "ingredient", "group_head",
    "OR", "U_case_not_ctrl", "V_ctrl_not_case", "discordant_total",
    "q_value_bh",
    "P_trigger_p_gt_0p5_pooled", "pooled_CI95_low_p", "pooled_CI95_high_p",
    "evidence_label",
    "loo_group_discordant", "prior_a_used", "prior_b_used"
]].head(50))

# -----------------------------
# Save outputs
# -----------------------------
if "OUT_DIR" in globals():
    out_dir = OUT_DIR
    if isinstance(out_dir, (str, os.PathLike)):
        os.makedirs(out_dir, exist_ok=True)

        ingredient_path = os.path.join(out_dir, "ingredient_enriched_pooled.csv")
        group_path      = os.path.join(out_dir, "group_stats.csv")
        final_path      = os.path.join(out_dir, "final_triggers.csv")

        ingredient_enriched.to_csv(ingredient_path, index=False)
        group_stats.to_csv(group_path, index=False)
        final_triggers.to_csv(final_path, index=False)

        print(f"\n Saved: {ingredient_path}")
        print(f"Saved: {group_path}")
        print(f"Saved: {final_path}")

# -----------------------------
# Mapping audit (coverage of grouping file)
# -----------------------------
mapping_keys = set(ingredient_to_group_norm.keys())
true_unmapped = ingredient_enriched[~ingredient_enriched["ingredient"].isin(mapping_keys)]

print("\n📌 Grouping audit")
print("Ingredients total:", len(ingredient_enriched))
print("Not present in ingredient_to_group keys (true unmapped):", len(true_unmapped))
if len(true_unmapped) > 0:
    display(true_unmapped[["ingredient"]].drop_duplicates().head(30))

print("\n Pooling settings used")
print({
    "POOL_LAMBDA": POOL_LAMBDA,
    "MIN_GROUP_LOO_DISCORDANT": MIN_GROUP_LOO_DISCORDANT,
    "BETA_PRIOR_A": BETA_PRIOR_A,
    "BETA_PRIOR_B": BETA_PRIOR_B,
    "POSTERIOR_MC_SAMPLES": POSTERIOR_MC_SAMPLES
})


 Group-level diagnostics built (for reporting/interpretation):


,group_head,group_U,group_V,group_discordant,group_n_members,group_OR,group_p_value,group_post_mean_p,group_CI95_low_p,group_CI95_high_p,group_P_gt_0p5,group_q_value_bh
0,milk,2,0,2,1,5.000000,0.50000,0.834570,0.341413,0.999773,0.92710,0.781250
1,yogurt,2,0,2,1,5.000000,0.50000,0.833434,0.327095,0.999808,0.92395,0.781250
2,butter,1,0,1,1,3.000000,1.00000,0.747398,0.141289,0.999583,0.81710,1.000000
3,cheese,2,2,4,1,1.000000,1.00000,0.498891,0.123086,0.875057,0.49950,1.000000
4,egg,3,4,7,1,0.777778,1.00000,0.438596,0.143379,0.769629,0.35735,1.000000
5,beans,2,3,5,1,0.714286,1.00000,0.416257,0.093493,0.789495,0.32950,1.000000
6,pasta,2,3,5,1,0.714286,1.00000,0.415791,0.093436,0.792416,0.32820,1.000000
7,oil,1,2,3,1,0.600000,1.00000,0.374914,0.038610,0.824317,0.28920,1.000000
8,cucumber,2,4,6,1,0.555556,0.68750,0.358629,0.077329,0.715390,0.21130,0.954861
9,flour,0,1,1,1,0.333333,1.00000,0.250281,0.000410,0.856569,0.17985,1.000000


Ingredient-level pooled results built (LOO hierarchical pooling):


,ingredient,U_case_not_ctrl,V_ctrl_not_case,discordant_total,eligible_strata,OR,p_value,p_case_only_posterior_mean,p_case_only_CI95_low,p_case_only_CI95_high,...,loo_group_V,loo_group_discordant,prior_a_used,prior_b_used,pooled_post_mean_p,pooled_CI95_low_p,pooled_CI95_high_p,P_trigger_p_gt_0p5_pooled,trigger_score,rank
0,milk,2,0,2,2,5.000000,0.50000,0.833191,0.336033,0.999761,...,0,0,0.5,0.5,0.834348,0.337222,0.999774,0.92805,0.92805,1
1,yogurt,2,0,2,2,5.000000,0.50000,0.833056,0.330622,0.999780,...,0,0,0.5,0.5,0.833855,0.342250,0.999828,0.92700,0.92700,2
2,butter,1,0,1,1,3.000000,1.00000,0.749288,0.140798,0.999541,...,0,0,0.5,0.5,0.751111,0.148708,0.999612,0.82045,0.82045,3
3,cheese,2,2,4,5,1.000000,1.00000,0.496782,0.119530,0.878525,...,0,0,0.5,0.5,0.500219,0.125086,0.880398,0.49730,0.49730,4
4,egg,3,4,7,8,0.777778,1.00000,0.437414,0.137297,0.764436,...,0,0,0.5,0.5,0.434888,0.140048,0.763517,0.34590,0.34590,5
5,pasta,2,3,5,5,0.714286,1.00000,0.413679,0.093987,0.786538,...,0,0,0.5,0.5,0.415841,0.094611,0.789793,0.32660,0.32660,6
6,beans,2,3,5,9,0.714286,1.00000,0.417240,0.093533,0.792447,...,0,0,0.5,0.5,0.414945,0.091920,0.784823,0.32380,0.32380,7
7,pure olive oil,1,2,3,8,0.600000,1.00000,0.375138,0.037994,0.821496,...,0,0,0.5,0.5,0.374891,0.038220,0.821216,0.28640,0.28640,8
8,cucumber,2,4,6,7,0.555556,0.68750,0.356406,0.076644,0.715196,...,0,0,0.5,0.5,0.359979,0.077904,0.715264,0.21220,0.21220,9
9,flour,0,1,1,1,0.333333,1.00000,0.251517,0.000365,0.854546,...,0,0,0.5,0.5,0.248966,0.000382,0.854559,0.18190,0.18190,10


Final triggers (q<= 0.1, discordant>= 6, P_trigger>=0.80): 0


,rank,ingredient,group_head,OR,U_case_not_ctrl,V_ctrl_not_case,discordant_total,q_value_bh,P_trigger_p_gt_0p5_pooled,pooled_CI95_low_p,pooled_CI95_high_p,evidence_label,loo_group_discordant,prior_a_used,prior_b_used



 Saved: ../data/processed/trigger_detection/ingredient_enriched_pooled.csv
Saved: ../data/processed/trigger_detection/group_stats.csv
Saved: ../data/processed/trigger_detection/final_triggers.csv

📌 Grouping audit
Ingredients total: 25
Not present in ingredient_to_group keys (true unmapped): 5


,ingredient
2,butter
6,beans
17,onion
18,pepper
23,oats



 Pooling settings used
{'POOL_LAMBDA': 0.1, 'MIN_GROUP_LOO_DISCORDANT': 6, 'BETA_PRIOR_A': 0.5, 'BETA_PRIOR_B': 0.5, 'POSTERIOR_MC_SAMPLES': 20000}
